# 21 · Исследовательский триаж газовых тревог (CPU, без обучения)

Метки описывают **гипотезы**, совместимые с поверкой или обслуживанием, но не подтверждают их. Инцидентных меток
нет. Тревоги не скрываются, backend и frozen G2 v3 не меняются. Контракт: `collab/013` + `collab/015`.

Выход: `results_21_gas_triage.json` и `summary_21_gas_triage_ru.md`, только агрегаты.


## Входы и запуск

1. Add Input: сырые `ext-journal-YYYY` и справочник каналов — те же источники, что у notebook 19.
2. Add Input: frozen output notebook 14 с `panel_manifest_v3.json` и parquet parts; SHA префикс `8cb6e0e2584f`.
3. Accelerator = None. Для проверки среды можно поставить `MODE = "SMOKE"` и выполнить Save & Run All.
4. Для результата по всем годам вернуть `MODE = "FULL"`, выполнить Save & Run All и скачать два итоговых файла.

FULL повторно читает архив: notebook 19 удалял временный секундный слой по умолчанию. На старте выводятся RAM,
свободное место и число бакетов. Полный прогон не запускается локально этим генератором.


In [ ]:
# ЕДИНСТВЕННАЯ ЯЧЕЙКА, КОТОРУЮ МОЖНО МЕНЯТЬ
MODE = "FULL"  # "FULL" или "SMOKE"; SMOKE не сравним по числам
print("Режим:", MODE)


In [ ]:
import os, sys, tempfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import polars as pl
print("python", sys.version.split()[0], "| polars", pl.__version__, "| numpy", np.__version__)


## Встроенный код


In [ ]:
SOURCES = {
"event_panel_v3.py": "\"\"\"Событийная панель v3: state machine по сырому журналу → панель (канал × сутки D) с раздельными целями.\n\nСпецификация согласована в переписке Claude ↔ GPT (Drive, документы 05 → 12) и docs/EVENT_PANEL_V3_RU.md.\nГлавные правила:\n- одна секунда канала = неупорядоченное множество значений; порядок строк CSV и ид_события порядком не считаются;\n- признаки на конец D — только прошлое; метка окна [D+2; D+3) может смотреть вперёд не дальше D+4 (L = 24 ч);\n- eligibility на конец D задаётся только состоянием на конец D; старт в D+1 — competing outcome, а не фильтр;\n- неоднозначное → null с причиной, а не 0;\n- цели разные и не смешиваются: T2a (onset «Неисправен» = потеря связи), T2b (потеря связи ≥ τ),\n  T2c (onset «Обесточен»), T2c_sustained, T1a (технические/диапазонные значения), T4 (газ ≥ 1 %).\nНичто здесь не является подтверждённым физическим отказом.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nSCHEMA_VERSION = \"3.0\"\nEVENT_COLUMNS = [\"ид_события\", \"ид_канала_данных\", \"дата\", \"время\", \"тревожное\", \"значение_датчика\"]\nCATALOGUE_COLUMNS = [\"ид_канала_данных\", \"тип_инж_системы\", \"тип_датчика\", \"ид_объект\"]\nKEY_SALT = \"ldt-misis-panel-v3:\"\n\nDEFAULT_CONFIG: dict[str, Any] = {\n    \"tau_minutes\": 5,                 # порог «устойчивого» эпизода; чувствительность 1/5/30/60 — отдельными прогонами\n    \"confirm_hours\": 24,              # L: самое позднее подтверждение метки, ч\n    \"active_lookback_days\": 7,        # строка (канал, D) есть, если канал слал события в [D-6; D]\n    \"tech_clean_days\": 7,             # T1a: onset только после ≥ 7 суток без технических значений\n    \"numeric_history_days\": 30,       # T1a: канал должен слать числа в последние 30 суток\n    \"window_offsets_days\": [7, 30],   # окна скользящих признаков\n    \"service_as_unknown\": True,       # «Неопределен» = U для связи (основная версия); False — sensitivity (GPT 16/Q1)\n    \"power_ok_from_function\": False,  # sensitivity: сообщения функции канала (норма/событие/число) считаются O питания\n}\n\n# заранее объявленные sensitivity-варианты разметки (выбор между ними не делается по validation-метрикам)\nSENSITIVITY_VARIANTS = {\"service_neutral\": {\"service_as_unknown\": False},\n                        \"power_ok_from_function\": {\"power_ok_from_function\": True}}\n\nLINK_STATES = (\"O\", \"F\", \"A\")  # ok / fault / ambiguous\nTARGETS = [\"target_t2a_link_onset\", \"target_t2b_link_sustained\", \"target_t2c_power_onset\",\n           \"target_t2c_power_sustained\", \"target_t1a_tech_value\", \"target_t4_gas_cross\"]\nBASE_FEATURES = [\"n_events\", \"n_alarm\", \"n_link_fault\", \"n_power_off\", \"n_service\", \"n_tech_like\", \"n_numeric\",\n                 \"n_multi_value_seconds\", \"num_mean\", \"num_min\", \"num_max\", \"num_std\", \"num_last\", \"gas_max\", \"gas_last\",\n                 \"link_state\", \"power_state\", \"d_weekday\", \"d_month\"]\n\n\n# ---------------------------------------------------------------------------\n# 0. Таксономия и ключи\n# ---------------------------------------------------------------------------\ndef load_taxonomy(path: Path) -> dict[str, Any]:\n    tax = json.loads(Path(path).read_text(encoding=\"utf-8\"))\n    seen: dict[str, str] = {}\n    for cls, values in tax[\"text_classes\"].items():\n        for v in values:\n            if v in seen and seen[v] != cls:\n                raise ValueError(f\"значение {v!r} в двух классах: {seen[v]} и {cls}\")\n            seen[v] = cls\n    tax[\"_value_to_class\"] = seen\n    tax[\"_sha256\"] = hashlib.sha256(Path(path).read_bytes()).hexdigest()\n    return tax\n\n\ndef pseudo_key(raw: str | None) -> str | None:\n    if raw is None:\n        return None\n    return hashlib.sha256((KEY_SALT + str(raw)).encode(\"utf-8\")).hexdigest()[:16]\n\n\ndef key_map(pl: Any, ids: Iterable[str], name: str) -> Any:\n    ids = sorted({str(i) for i in ids if i is not None})\n    return pl.DataFrame({\"raw\": ids, name: [pseudo_key(i) for i in ids]})\n\n\n# ---------------------------------------------------------------------------\n# 1. Классификация событий\n# ---------------------------------------------------------------------------\ndef classify_events(pl: Any, frame: Any, tax: dict[str, Any], catalogue: Any) -> Any:\n    \"\"\"frame: сырые колонки журнала (строки). Возвращает ch, t, v, alarm, num, cls, is_gas, тип_*.\"\"\"\n    num_cfg, date_cfg = tax[\"numeric\"], tax[\"date_like\"]\n    value = pl.col(\"значение_датчика\").fill_null(\"\")\n    num = value.str.replace(\",\", \".\", literal=True).cast(pl.Float64, strict=False)\n    t = pl.concat_str([\"дата\", \"время\"], separator=\" \").str.strptime(pl.Datetime(\"us\"), \"%Y-%m-%d %H:%M:%S\", strict=False)\n    out = (\n        frame.join(catalogue, on=\"ид_канала_данных\", how=\"left\")\n        .with_columns(\n            pl.col(\"ид_канала_данных\").alias(\"ch\"), t.alias(\"t\"), value.alias(\"v\"), num.alias(\"num\"),\n            pl.col(\"тревожное\").fill_null(\"\").str.to_lowercase().is_in([\"t\", \"true\"]).alias(\"alarm\"),\n            (pl.col(\"тип_датчика\") == num_cfg[\"gas_sensor_type\"]).fill_null(False).alias(\"is_gas\"),\n        )\n        .filter(pl.col(\"t\").is_not_null())\n    )\n    is_date = pl.col(\"v\").str.contains(date_cfg[\"regex\"])\n    parsed_date = pl.col(\"v\").str.slice(0, 19).str.strptime(pl.Datetime(\"us\"), \"%d.%m.%Y %H:%M:%S\", strict=False)\n    tol = int(date_cfg[\"duplicate_timestamp_tolerance_seconds\"])\n    text_cls = pl.col(\"v\").replace_strict(tax[\"_value_to_class\"], default=\"unknown_value\", return_dtype=pl.String)\n    cls = (\n        pl.when(pl.col(\"num\").is_not_null() & pl.col(\"num\").is_in(num_cfg[\"technical_value_candidates\"])).then(pl.lit(\"tech_value\"))\n        .when(pl.col(\"num\").is_not_null() & pl.col(\"is_gas\")\n              & ((pl.col(\"num\") < num_cfg[\"gas_out_of_range\"][\"below\"]) | (pl.col(\"num\") > num_cfg[\"gas_out_of_range\"][\"above\"])))\n        .then(pl.lit(\"gas_out_of_range\"))\n        .when(pl.col(\"num\").is_not_null()).then(pl.lit(\"numeric\"))\n        .when(is_date & pl.col(\"v\").str.starts_with(date_cfg[\"clock_1970_prefix\"])).then(pl.lit(\"clock_1970\"))\n        .when(is_date & ((parsed_date - pl.col(\"t\")).dt.total_seconds().abs() <= tol)).then(pl.lit(\"dup_timestamp\"))\n        .when(is_date).then(pl.lit(\"date_other\"))\n        .when(pl.col(\"v\") == \"\").then(pl.lit(\"empty_value\"))\n        .otherwise(text_cls)\n    )\n    return out.with_columns(cls.alias(\"cls\")).select(\n        \"ch\", \"t\", \"v\", \"alarm\", \"num\", \"cls\", \"is_gas\", \"тип_датчика\", \"тип_инж_системы\", \"ид_объект\")\n\n\n# ---------------------------------------------------------------------------\n# 2. Секундный уровень и точки смены состояния (RLE)\n# ---------------------------------------------------------------------------\ndef second_table(pl: Any, ev: Any, tax: dict[str, Any]) -> Any:\n    \"\"\"Одна строка = (канал, секунда). Порядок внутри секунды не используется.\n    Для каждого потока (связь, питание) секунда: F — только отказ, O — только рабочее состояние того же потока,\n    A — отказ и O вместе или любое неизвестное (U) этого потока; null — секунда не несёт сведений о потоке.\"\"\"\n    streams = tax[\"stream_mapping\"][\"streams\"]\n    aggs = [pl.col(\"v\").n_unique().alias(\"n_values\"), (pl.col(\"cls\") == \"link_fault\").any().alias(\"has_fault\")]\n    for name, st in streams.items():\n        aggs += [pl.col(\"cls\").is_in(st[\"fault\"]).any().alias(f\"_{name}_f\"),\n                 pl.col(\"cls\").is_in(st[\"ok\"]).any().alias(f\"_{name}_o\"),\n                 pl.col(\"cls\").is_in(st[\"unknown\"]).any().alias(f\"_{name}_u\")]\n    sec = ev.group_by(\"ch\", \"t\").agg(*aggs)\n    cols = []\n    for name in streams:\n        f, o, u = pl.col(f\"_{name}_f\"), pl.col(f\"_{name}_o\"), pl.col(f\"_{name}_u\")\n        override = bool(streams[name].get(\"unknown_overrides\", True))\n        u_wins = u if override else (u & ~f & ~o)\n        cols.append(pl.when(f & o).then(pl.lit(\"A\")).when(u_wins).then(pl.lit(\"A\"))\n                    .when(f).then(pl.lit(\"F\")).when(o).then(pl.lit(\"O\")).otherwise(None).alias(f\"{name}_cls\"))\n        cols.append(pl.when(f & o).then(pl.lit(\"conflict\")).when(u_wins).then(pl.lit(\"unknown\")).otherwise(None)\n                    .alias(f\"{name}_a_cause\"))\n    return sec.with_columns(cols).drop([c for c in sec.columns if c.startswith(\"_\")])\n\n\ndef change_points(pl: Any, sec: Any, cls_col: str) -> Any:\n    \"\"\"RLE решающих секунд: строки, где класс сменился (по каналу, по времени).\"\"\"\n    d = sec.filter(pl.col(cls_col).is_not_null()).select(\"ch\", \"t\", pl.col(cls_col).alias(\"cls\")).sort(\"ch\", \"t\")\n    return rle(pl, d)\n\n\ndef rle(pl: Any, d: Any) -> Any:\n    d = d.sort(\"ch\", \"t\")\n    prev = pl.col(\"cls\").shift(1).over(\"ch\")\n    return d.filter(prev.is_null() | (pl.col(\"cls\") != prev)).select(\"ch\", \"t\", \"cls\")\n\n\n# ---------------------------------------------------------------------------\n# 3. Эпизоды\n# ---------------------------------------------------------------------------\ndef episodes(pl: Any, cp: Any, data_end: datetime, tau_minutes: float) -> Any:\n    \"\"\"Эпизод = RLE-блок класса F. onset_type: strict (перед ним однозначное O) или ambiguous.\n    Длительность: до следующей точки смены. Следующая O → exact; A или конец данных → цензура (нижняя граница).\n    sustained = 1 только при однозначном закрытии (O того же потока) после t0 + τ: повторы «Неисправен» и тишина\n    непрерывность не доказывают (GPT 14, C7); незакрытый эпизод → null.\"\"\"\n    cp = cp.sort(\"ch\", \"t\").with_columns(\n        pl.col(\"cls\").shift(1).over(\"ch\").alias(\"prev_cls\"),\n        pl.col(\"cls\").shift(-1).over(\"ch\").alias(\"next_cls\"),\n        pl.col(\"t\").shift(-1).over(\"ch\").alias(\"next_t\"),\n    )\n    tau_s = tau_minutes * 60.0\n    ep = cp.filter(pl.col(\"cls\") == \"F\").with_columns(\n        pl.when(pl.col(\"prev_cls\") == \"O\").then(pl.lit(\"strict\")).otherwise(pl.lit(\"ambiguous\")).alias(\"onset_type\"),\n        (pl.col(\"next_cls\") == \"O\").fill_null(False).alias(\"exact\"),\n        pl.col(\"next_t\").fill_null(pl.lit(data_end)).alias(\"end_t\"),\n    ).with_columns(\n        (pl.col(\"end_t\") - pl.col(\"t\")).dt.total_seconds().cast(pl.Float64).alias(\"duration_s\"),\n    ).with_columns(\n        pl.when(pl.col(\"exact\") & (pl.col(\"duration_s\") >= tau_s)).then(pl.lit(1, pl.Int8))\n        .when(pl.col(\"exact\")).then(pl.lit(0, pl.Int8))\n        .otherwise(None).alias(\"sustained\"),\n    )\n    return ep.select(\"ch\", pl.col(\"t\").alias(\"onset_t\"), \"onset_type\", \"exact\", \"end_t\", \"duration_s\", \"sustained\")\n\n\n# ---------------------------------------------------------------------------\n# 4. Дневные агрегаты\n# ---------------------------------------------------------------------------\nCOUNT_CLASSES = [\"link_fault\", \"power_off\", \"power_on\", \"power_degraded\", \"device_off\", \"service\",\n                 \"undefined_measurement\", \"object_summary\", \"security_mode\", \"flood\", \"event_alarm\", \"normal\",\n                 \"numeric\", \"tech_value\", \"gas_out_of_range\", \"clock_1970\", \"dup_timestamp\", \"date_other\",\n                 \"unknown_value\", \"empty_value\"]\n\n\ndef daily_aggregates(pl: Any, ev: Any, sec: Any, tax: dict[str, Any]) -> Any:\n    thr = tax[\"numeric\"][\"gas_threshold_percent\"]\n    low = tax[\"numeric\"][\"gas_band_low\"]\n    warn = tax[\"numeric\"][\"gas_warning_percent\"]\n    # порядок внутри группы = время; в одной секунде значения не упорядочены — берём детерминированно (по значению)\n    ev = ev.sort(\"ch\", \"t\", \"num\", nulls_last=False, maintain_order=True).with_columns(pl.col(\"t\").dt.date().alias(\"day\"))\n    numeric = pl.col(\"cls\") == \"numeric\"\n    gas = numeric & pl.col(\"is_gas\")\n    agg = ev.group_by(\"ch\", \"day\").agg(\n        pl.len().alias(\"n_events\"),\n        pl.col(\"alarm\").sum().alias(\"n_alarm\"),\n        *[(pl.col(\"cls\") == c).sum().alias(f\"n_{c}\") for c in COUNT_CLASSES],\n        pl.col(\"num\").filter(numeric).mean().alias(\"num_mean\"),\n        pl.col(\"num\").filter(numeric).min().alias(\"num_min\"),\n        pl.col(\"num\").filter(numeric).max().alias(\"num_max\"),\n        pl.col(\"num\").filter(numeric).std().alias(\"num_std\"),\n        pl.col(\"num\").filter(numeric).last().alias(\"num_last\"),\n        pl.col(\"num\").filter(gas).max().alias(\"gas_max\"),\n        pl.col(\"num\").filter(gas).count().alias(\"gas_n\"),\n        (gas & (pl.col(\"num\") >= low) & (pl.col(\"num\") < thr)).sum().alias(\"gas_band_n\"),\n        (gas & (pl.col(\"num\") >= thr)).sum().alias(\"gas_ge_thr_n\"),\n        (gas & (pl.col(\"num\") >= warn)).sum().alias(\"gas_ge_warn_n\"),\n        pl.col(\"num\").filter(gas).last().alias(\"gas_last\"),\n        # последняя газовая секунда суток содержит значения по обе стороны порога → сторона на конец D неизвестна (GPT 42)\n        ((pl.col(\"num\").filter(gas & (pl.col(\"t\") == pl.col(\"t\").filter(gas).max())).min() < thr)\n         & (pl.col(\"num\").filter(gas & (pl.col(\"t\") == pl.col(\"t\").filter(gas).max())).max() >= thr)).alias(\"gas_last_mixed\"),\n    )\n    s = sec.with_columns(pl.col(\"t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n        (pl.col(\"n_values\") > 1).sum().alias(\"n_multi_value_seconds\"),\n        (pl.col(\"link_cls\") == \"F\").sum().alias(\"n_link_fault_seconds\"),\n        (pl.col(\"link_cls\") == \"A\").sum().alias(\"n_link_ambiguous_seconds\"),\n        (pl.col(\"power_cls\") == \"F\").sum().alias(\"n_power_off_seconds\"),\n    )\n    return agg.join(s, on=[\"ch\", \"day\"], how=\"left\")\n\n\ndef gas_crossings(pl: Any, ev: Any, carry: Any, thr: float) -> tuple[Any, Any]:\n    \"\"\"Строгое пересечение порога на секундном уровне (секунда = множество значений без порядка, как для состояний):\n    strict — все показания секунды ≥ thr, а все показания предыдущей секунды того же канала < thr;\n    неоднозначно (cross_unknown) — в секунде есть значения по обе стороны порога, либо предыдущая секунда такая,\n    либо предыдущего показания нет. Раньше показания одной секунды сортировались по значению, и пара 0 / 2,55\n    в одну секунду давала ложный strict crossing (GPT 37).\n    carry: последняя газовая секунда каждого канала из предыдущего года (ch, t, mn, mx).\"\"\"\n    g = (ev.filter((pl.col(\"cls\") == \"numeric\") & pl.col(\"is_gas\"))\n         .group_by(\"ch\", \"t\").agg(pl.col(\"num\").min().alias(\"mn\"), pl.col(\"num\").max().alias(\"mx\")))\n    both = pl.concat([carry.select(\"ch\", \"t\", \"mn\", \"mx\").with_columns(pl.lit(True).alias(\"_carry\")),\n                      g.with_columns(pl.lit(False).alias(\"_carry\"))], how=\"vertical\").sort(\"ch\", \"t\", \"_carry\",\n                                                                                          descending=[False, False, True])\n    pmn, pmx = pl.col(\"mn\").shift(1).over(\"ch\"), pl.col(\"mx\").shift(1).over(\"ch\")\n    straddle = (pl.col(\"mn\") < thr) & (pl.col(\"mx\") >= thr)\n    prev_straddle = (pmn < thr) & (pmx >= thr)\n    x = both.with_columns(\n        ((pl.col(\"mn\") >= thr) & (pmx < thr)).fill_null(False).alias(\"cross\"),\n        ((pl.col(\"mx\") >= thr) & (pmx.is_null() | straddle | ((pl.col(\"mn\") >= thr) & prev_straddle.fill_null(False))))\n        .fill_null(False).alias(\"cross_unknown_prev\"))\n    x = x.filter(~pl.col(\"_carry\"))\n    daily = x.with_columns(pl.col(\"t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n        pl.col(\"cross\").sum().alias(\"gas_cross_n\"), pl.col(\"cross_unknown_prev\").sum().alias(\"gas_cross_unknown_n\"))\n    new_carry = (pl.concat([carry.select(\"ch\", \"t\", \"mn\", \"mx\"), g], how=\"vertical\").sort(\"ch\", \"t\")\n                 .group_by(\"ch\", maintain_order=True).agg(pl.col(\"t\").last(), pl.col(\"mn\").last(), pl.col(\"mx\").last()))\n    return daily, new_carry\n\n\n# ---------------------------------------------------------------------------\n# 5. Сборка панели\n# ---------------------------------------------------------------------------\ndef _shift_days(pl: Any, frame: Any, cols: list[str], offset: int, suffix: str) -> Any:\n    \"\"\"Факты дня X → строки D = X − offset.\"\"\"\n    return frame.select(\"ch\", (pl.col(\"day\") - pl.duration(days=offset)).alias(\"day\"),\n                        *[pl.col(c).alias(f\"{c}{suffix}\") for c in cols])\n\n\ndef build_panel_from_parts(pl: Any, daily: Any, link_cp: Any, power_cp: Any, gas_daily: Any,\n                           meta: Any, data_end: datetime, config: dict[str, Any]) -> tuple[Any, dict[str, Any]]:\n    tau = float(config[\"tau_minutes\"])\n    L = int(config[\"confirm_hours\"])\n    lookback = int(config[\"active_lookback_days\"])\n    data_end_day = data_end.date()\n\n    link_ep = episodes(pl, link_cp, data_end, tau)\n    power_ep = episodes(pl, power_cp, data_end, tau)\n\n    daily = daily.join(gas_daily, on=[\"ch\", \"day\"], how=\"left\").with_columns(\n        pl.col(\"gas_cross_n\").fill_null(0), pl.col(\"gas_cross_unknown_n\").fill_null(0))\n\n    # грид: (канал, D), если канал слал события в [D-lookback+1; D]\n    grid = (daily.select(\"ch\", \"day\")\n            .with_columns(pl.int_ranges(0, lookback).alias(\"_k\")).explode(\"_k\")\n            .with_columns((pl.col(\"day\") + pl.duration(days=pl.col(\"_k\"))).alias(\"day\"))\n            .filter(pl.col(\"day\") <= pl.lit(data_end_day))\n            .select(\"ch\", \"day\").unique().sort(\"ch\", \"day\"))\n\n    count_cols = [c for c in daily.columns if c.startswith(\"n_\") or c in (\"gas_n\", \"gas_band_n\", \"gas_ge_thr_n\",\n                                                                               \"gas_ge_warn_n\", \"gas_cross_n\")]\n    panel = grid.join(daily, on=[\"ch\", \"day\"], how=\"left\").with_columns(\n        [pl.col(c).fill_null(0) for c in count_cols]).sort(\"ch\", \"day\")\n\n    # эпизоды по дням: onset-день (для меток и признаков) и день завершения (для признаков «прошлых эпизодов»)\n    def per_day(ep: Any, prefix: str, cp: Any) -> tuple[Any, Any, Any]:\n        on = ep.with_columns(pl.col(\"onset_t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n            (pl.col(\"onset_type\") == \"strict\").sum().alias(f\"{prefix}_onset_strict\"),\n            (pl.col(\"onset_type\") == \"ambiguous\").sum().alias(f\"{prefix}_onset_ambiguous\"),\n            ((pl.col(\"onset_type\") == \"strict\") & (pl.col(\"sustained\") == 1)).sum().alias(f\"{prefix}_onset_sustained\"),\n            ((pl.col(\"onset_type\") == \"strict\") & (pl.col(\"sustained\") == 0)).sum().alias(f\"{prefix}_onset_transient\"),\n            ((pl.col(\"onset_type\") == \"strict\") & pl.col(\"sustained\").is_null()).sum().alias(f\"{prefix}_onset_censored\"),\n        )\n        done = ep.filter(pl.col(\"exact\")).with_columns(pl.col(\"end_t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n            (pl.col(\"sustained\") == 1).sum().alias(f\"{prefix}_completed_long\"),\n            (pl.col(\"sustained\") == 0).sum().alias(f\"{prefix}_completed_short\"),\n        )\n        # неоднозначные секунды (A-блоки): переход мог быть скрыт — окно с таким блоком не отрицательное (GPT 14, C2/C12)\n        amb = cp.filter(pl.col(\"cls\") == \"A\").with_columns(pl.col(\"t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n            pl.len().cast(pl.Int64).alias(f\"{prefix}_ambiguous_block\"))\n        return on, done, amb\n\n    link_on, link_done, link_amb = per_day(link_ep, \"link\", link_cp)\n    power_on, power_done, power_amb = per_day(power_ep, \"power\", power_cp)\n    for part in (link_on, link_done, power_on, power_done, link_amb, power_amb):\n        cols = [c for c in part.columns if c not in (\"ch\", \"day\")]\n        panel = panel.join(part, on=[\"ch\", \"day\"], how=\"left\").with_columns([pl.col(c).fill_null(0) for c in cols])\n\n    # скользящие признаки (только прошлое, включая D)\n    roll_src = [\"n_events\", \"n_alarm\", \"n_link_fault\", \"n_power_off\", \"n_service\", \"n_device_off\",\n                \"n_undefined_measurement\", \"n_tech_value\", \"n_gas_out_of_range\", \"n_clock_1970\", \"n_dup_timestamp\",\n                \"n_date_other\", \"n_unknown_value\", \"n_flood\", \"n_event_alarm\", \"n_numeric\", \"n_multi_value_seconds\",\n                \"n_link_fault_seconds\", \"n_link_ambiguous_seconds\", \"n_power_off_seconds\",\n                \"link_onset_strict\", \"link_onset_ambiguous\", \"power_onset_strict\",\n                \"link_completed_long\", \"link_completed_short\", \"power_completed_long\", \"power_completed_short\",\n                \"gas_n\", \"gas_band_n\", \"gas_ge_thr_n\", \"gas_ge_warn_n\", \"gas_cross_n\"]\n    panel = panel.with_columns((pl.col(\"n_events\") > 0).cast(pl.Int32).alias(\"active_day\"),\n                               (pl.col(\"n_tech_value\") + pl.col(\"n_gas_out_of_range\") + pl.col(\"n_undefined_measurement\"))\n                               .alias(\"n_tech_like\"),\n                               (pl.col(\"n_numeric\") + pl.col(\"n_undefined_measurement\") + pl.col(\"n_tech_value\")\n                                + pl.col(\"n_gas_out_of_range\")).alias(\"n_measurement\"))\n    roll_exprs = []\n    for w in config[\"window_offsets_days\"]:\n        for c in roll_src + [\"active_day\", \"n_tech_like\", \"n_measurement\"]:\n            # polars < 1.3x не принимает null во входе rolling_*_by: счётчики без событий = 0\n            roll_exprs.append(pl.col(c).fill_null(0).rolling_sum_by(\"day\", window_size=f\"{w}d\", closed=\"right\")\n                              .over(\"ch\").alias(f\"f_{c}_{w}d\"))\n        # максимум газа: дни без газовых чисел → −∞ во входе, окно без чисел → null на выходе\n        gmax = pl.col(\"gas_max\").fill_null(float(\"-inf\")).rolling_max_by(\"day\", window_size=f\"{w}d\", closed=\"right\").over(\"ch\")\n        roll_exprs.append(pl.when(gmax == float(\"-inf\")).then(None).otherwise(gmax).alias(f\"f_gas_max_{w}d\"))\n    panel = panel.sort(\"ch\", \"day\").with_columns(roll_exprs)\n    # последнее газовое/числовое значение на конец D (forward fill по грид-строкам канала)\n    panel = panel.with_columns(pl.col(\"gas_last\").forward_fill().over(\"ch\").alias(\"f_gas_last\"),\n                               pl.col(\"num_last\").forward_fill().over(\"ch\").alias(\"f_num_last\"),\n                               pl.when(pl.col(\"gas_last\").is_not_null()).then(pl.col(\"gas_last_mixed\").fill_null(False))\n                               .forward_fill().over(\"ch\").fill_null(False).alias(\"_gas_last_mixed\"))\n\n    # состояния на конец D (as-of по точкам смены)\n    cutoff = (pl.col(\"day\").cast(pl.Datetime(\"us\")) + pl.duration(days=1) - pl.duration(microseconds=1))\n    panel = panel.with_columns(cutoff.alias(\"_cutoff_t\")).sort(\"_cutoff_t\")\n    for name, cp in ((\"link\", link_cp), (\"power\", power_cp)):\n        c = cp.select(\"ch\", pl.col(\"t\").alias(f\"_{name}_t\"), pl.col(\"cls\").alias(f\"{name}_state\")).sort(f\"_{name}_t\")\n        panel = panel.join_asof(c, left_on=\"_cutoff_t\", right_on=f\"_{name}_t\", by=\"ch\", strategy=\"backward\")\n        panel = panel.with_columns(\n            ((pl.col(\"_cutoff_t\") - pl.col(f\"_{name}_t\")).dt.total_seconds() / 86400.0).alias(f\"f_{name}_state_age_days\"))\n        last_on = (link_ep if name == \"link\" else power_ep).filter(pl.col(\"onset_type\") == \"strict\").select(\n            \"ch\", pl.col(\"onset_t\").alias(f\"_{name}_last_onset\")).sort(f\"_{name}_last_onset\")\n        panel = panel.join_asof(last_on, left_on=\"_cutoff_t\", right_on=f\"_{name}_last_onset\", by=\"ch\",\n                                strategy=\"backward\").with_columns(\n            ((pl.col(\"_cutoff_t\") - pl.col(f\"_{name}_last_onset\")).dt.total_seconds() / 86400.0)\n            .alias(f\"f_days_since_{name}_onset\"))\n    panel = panel.sort(\"ch\", \"day\")\n\n    # факты будущих дней для меток\n    fact_cols = [\"n_events\", \"link_onset_strict\", \"link_onset_ambiguous\", \"link_onset_sustained\", \"link_onset_censored\",\n                 \"link_onset_transient\", \"power_onset_strict\", \"power_onset_ambiguous\", \"power_onset_sustained\",\n                 \"power_onset_censored\", \"power_onset_transient\", \"link_ambiguous_block\", \"power_ambiguous_block\",\n                 \"n_tech_like\", \"n_measurement\", \"gas_n\",\n                 \"gas_cross_n\", \"gas_cross_unknown_n\"]\n    # каждый наблюдённый день есть в гриде (k = 0), поэтому факты любого дня с событиями — в panel\n    facts = panel.select(\"ch\", \"day\", *fact_cols)\n    d1 = _shift_days(pl, facts, [\"link_onset_strict\", \"link_onset_ambiguous\", \"power_onset_strict\",\n                                 \"power_onset_ambiguous\", \"gas_cross_n\"], 1, \"_d1\")\n    d2 = _shift_days(pl, facts, fact_cols, 2, \"_d2\")\n    d3 = _shift_days(pl, facts, [\"n_events\"], 3, \"_d3\")\n    panel = panel.join(d1, on=[\"ch\", \"day\"], how=\"left\").join(d2, on=[\"ch\", \"day\"], how=\"left\") \\\n                 .join(d3, on=[\"ch\", \"day\"], how=\"left\")\n    fut_cols = [c for c in panel.columns if c.endswith((\"_d1\", \"_d2\", \"_d3\"))]\n    panel = panel.with_columns([pl.col(c).fill_null(0) for c in fut_cols])\n\n    in_data = (pl.col(\"day\") + pl.duration(days=3)) <= pl.lit(data_end_day)  # окно + L внутри данных\n    observed = in_data & ((pl.col(\"n_events_d2\") + pl.col(\"n_events_d3\")) > 0)\n\n    def onset_target(prefix: str, state_col: str, sustained: bool) -> tuple[Any, Any]:\n        strict = pl.col(f\"{prefix}_onset_strict_d2\")\n        amb = pl.col(f\"{prefix}_onset_ambiguous_d2\") + pl.col(f\"{prefix}_ambiguous_block_d2\")\n        eligible = pl.col(state_col) == \"O\"\n        if not sustained:\n            label = (pl.when(~eligible.fill_null(False)).then(None)\n                     .when(~in_data).then(None)\n                     .when(strict > 0).then(pl.lit(1))\n                     .when(amb > 0).then(None)\n                     .when(observed).then(pl.lit(0))\n                     .otherwise(None))\n            reason = (pl.when(pl.col(state_col).is_null()).then(pl.lit(\"no_state_history\"))\n                      .when(~eligible).then(pl.lit(\"state_not_ok_at_D\"))\n                      .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                      .when(strict > 0).then(pl.lit(\"positive\"))\n                      .when(amb > 0).then(pl.lit(\"ambiguous_onset\"))\n                      .when(observed).then(pl.lit(\"negative\"))\n                      .otherwise(pl.lit(\"unobserved_window\")))\n        else:\n            sus, cen, tra = (pl.col(f\"{prefix}_onset_sustained_d2\"), pl.col(f\"{prefix}_onset_censored_d2\"),\n                             pl.col(f\"{prefix}_onset_transient_d2\"))\n            label = (pl.when(~eligible.fill_null(False)).then(None)\n                     .when(~in_data).then(None)\n                     .when(sus > 0).then(pl.lit(1))\n                     .when(cen > 0).then(None)\n                     .when(amb > 0).then(None)\n                     .when(observed).then(pl.lit(0))  # включая только кратковременные (transient) onset\n                     .otherwise(None))\n            reason = (pl.when(pl.col(state_col).is_null()).then(pl.lit(\"no_state_history\"))\n                      .when(~eligible).then(pl.lit(\"state_not_ok_at_D\"))\n                      .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                      .when(sus > 0).then(pl.lit(\"positive\"))\n                      .when(cen > 0).then(pl.lit(\"censored_duration\"))\n                      .when(amb > 0).then(pl.lit(\"ambiguous_onset\"))\n                      .when(observed & (tra > 0)).then(pl.lit(\"negative_transient_only\"))\n                      .when(observed).then(pl.lit(\"negative\"))\n                      .otherwise(pl.lit(\"unobserved_window\")))\n        return label.cast(pl.Int8), reason\n\n    targets = {}\n    reasons = {}\n    for name, prefix, state, sus in ((\"target_t2a_link_onset\", \"link\", \"link_state\", False),\n                                     (\"target_t2b_link_sustained\", \"link\", \"link_state\", True),\n                                     (\"target_t2c_power_onset\", \"power\", \"power_state\", False),\n                                     (\"target_t2c_power_sustained\", \"power\", \"power_state\", True)):\n        lab, rea = onset_target(prefix, state, sus)\n        targets[name], reasons[name] = lab, rea\n\n    # T1a: технические/диапазонные значения после ≥ tech_clean_days чистых суток\n    clean = int(config[\"tech_clean_days\"])\n    hist = int(config[\"numeric_history_days\"])\n    panel = panel.with_columns(\n        pl.col(\"n_tech_like\").fill_null(0).rolling_sum_by(\"day\", window_size=f\"{clean}d\", closed=\"right\").over(\"ch\").alias(\"_tech_recent\"),\n        pl.col(\"n_measurement\").fill_null(0).rolling_sum_by(\"day\", window_size=f\"{hist}d\", closed=\"right\").over(\"ch\").alias(\"_meas_hist\"),\n    )\n    t1_elig = (pl.col(\"_meas_hist\") > 0) & (pl.col(\"_tech_recent\") == 0)\n    targets[\"target_t1a_tech_value\"] = (pl.when(~t1_elig | ~in_data).then(None)\n                                        .when(pl.col(\"n_tech_like_d2\") > 0).then(pl.lit(1))\n                                        .when(pl.col(\"n_measurement_d2\") > 0).then(pl.lit(0))\n                                        .otherwise(None)).cast(pl.Int8)\n    reasons[\"target_t1a_tech_value\"] = (pl.when(pl.col(\"_meas_hist\") == 0).then(pl.lit(\"no_measurements_30d\"))\n                                        .when(pl.col(\"_tech_recent\") > 0).then(pl.lit(\"recent_tech_value\"))\n                                        .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                                        .when(pl.col(\"n_tech_like_d2\") > 0).then(pl.lit(\"positive\"))\n                                        .when(pl.col(\"n_measurement_d2\") > 0).then(pl.lit(\"negative\"))\n                                        .otherwise(pl.lit(\"unobserved_window\")))\n\n    # T4: газ — строгое пересечение 1 % в окне; канал газовый, последнее показание на конец D < порога\n    thr = config[\"gas_threshold_percent\"]\n    gas_elig = (pl.col(\"f_gas_n_7d\") > 0) & (pl.col(\"f_gas_last\") < thr) & ~pl.col(\"_gas_last_mixed\")\n    targets[\"target_t4_gas_cross\"] = (pl.when(~gas_elig.fill_null(False) | ~in_data).then(None)\n                                      .when(pl.col(\"gas_cross_n_d2\") > 0).then(pl.lit(1))\n                                      .when(pl.col(\"gas_cross_unknown_n_d2\") > 0).then(None)\n                                      .when(pl.col(\"gas_n_d2\") > 0).then(pl.lit(0))\n                                      .otherwise(None)).cast(pl.Int8)\n    reasons[\"target_t4_gas_cross\"] = (pl.when(pl.col(\"f_gas_n_7d\").fill_null(0) == 0).then(pl.lit(\"not_gas_stream\"))\n                                      .when(pl.col(\"_gas_last_mixed\")).then(pl.lit(\"ambiguous_last_reading_at_D\"))\n                                      .when(pl.col(\"f_gas_last\") >= thr).then(pl.lit(\"above_threshold_at_D\"))\n                                      .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                                      .when(pl.col(\"gas_cross_n_d2\") > 0).then(pl.lit(\"positive\"))\n                                      .when(pl.col(\"gas_cross_unknown_n_d2\") > 0).then(pl.lit(\"unknown_previous_reading\"))\n                                      .when(pl.col(\"gas_n_d2\") > 0).then(pl.lit(\"negative\"))\n                                      .otherwise(pl.lit(\"unobserved_window\")))\n\n    panel = panel.with_columns(\n        **targets, **{f\"reason_{k[len('target_'):]}\": v for k, v in reasons.items()},\n        competing_link_onset_d1=(pl.col(\"link_onset_strict_d1\") + pl.col(\"link_onset_ambiguous_d1\") > 0),\n        competing_power_onset_d1=(pl.col(\"power_onset_strict_d1\") + pl.col(\"power_onset_ambiguous_d1\") > 0),\n        competing_gas_cross_d1=(pl.col(\"gas_cross_n_d1\") > 0),\n    )\n\n    # мета: тип, объект, псевдоним ключей; календарь\n    panel = panel.join(meta, on=\"ch\", how=\"left\").with_columns(\n        pl.col(\"day\").alias(\"d_cutoff_date\"),\n        (pl.col(\"day\") + pl.duration(days=2)).alias(\"d_target_start_date\"),\n        (pl.col(\"day\") + pl.duration(days=3)).alias(\"d_target_end_date_exclusive\"),\n        (pl.col(\"day\") + pl.duration(days=3) + pl.duration(hours=L)).alias(\"d_label_decision_end\"),\n        pl.col(\"day\").dt.year().alias(\"d_year\"),\n        pl.col(\"day\").dt.weekday().alias(\"d_weekday\"),\n        pl.col(\"day\").dt.month().alias(\"d_month\"),\n    )\n    keys = key_map(pl, panel[\"ch\"].unique().to_list(), \"d_channel_key\")\n    objs = key_map(pl, panel[\"ид_объект\"].drop_nulls().unique().to_list(), \"d_object_key\")\n    panel = (panel.join(keys, left_on=\"ch\", right_on=\"raw\", how=\"left\")\n             .join(objs, left_on=\"ид_объект\", right_on=\"raw\", how=\"left\"))\n\n    feature_cols = [c for c in panel.columns if c.startswith(\"f_\")] + BASE_FEATURES\n    target_cols = list(targets)\n    assert target_cols == TARGETS\n    keep = ([\"d_channel_key\", \"d_object_key\", \"d_cutoff_date\", \"d_target_start_date\", \"d_target_end_date_exclusive\",\n             \"d_label_decision_end\", \"d_year\", \"тип_датчика\", \"тип_инж_системы\"] + feature_cols + target_cols\n            + [f\"reason_{k[len('target_'):]}\" for k in target_cols]\n            + [\"competing_link_onset_d1\", \"competing_power_onset_d1\", \"competing_gas_cross_d1\"])\n    out = panel.select(keep).sort(\"d_channel_key\", \"d_cutoff_date\")\n\n    durations = pl.concat([\n        e.select(pl.lit(n).alias(\"kind\"), \"onset_type\", \"exact\", \"duration_s\", \"sustained\")\n        for n, e in ((\"link\", link_ep), (\"power\", power_ep))], how=\"vertical\")\n    return out, {\"rows\": out.height, \"channels\": out[\"d_channel_key\"].n_unique(), \"_episode_durations\": durations}\n\n\n# ---------------------------------------------------------------------------\n# 6. Аудит (только агрегаты)\n# ---------------------------------------------------------------------------\ndef final_audit(pl: Any, panel_lf: Any, durations: Any) -> dict[str, Any]:\n    def ep_stats(kind: str) -> dict[str, Any]:\n        ep = durations.filter(pl.col(\"kind\") == kind)\n        s = ep.filter(pl.col(\"onset_type\") == \"strict\")\n        dur = s.filter(pl.col(\"exact\"))[\"duration_s\"]\n        return {\n            \"episodes\": ep.height, \"strict_onsets\": s.height,\n            \"ambiguous_onsets\": int((ep[\"onset_type\"] == \"ambiguous\").sum()),\n            \"strict_exact\": dur.len(), \"strict_censored\": int((~s[\"exact\"]).sum()),\n            \"exact_duration_s_quantiles\": ({str(q): float(dur.quantile(q)) for q in (0.1, 0.5, 0.9, 0.99)} if dur.len() else None),\n            \"sustained_1\": int((s[\"sustained\"] == 1).sum()), \"sustained_0\": int((s[\"sustained\"] == 0).sum()),\n            \"sustained_null\": int(s[\"sustained\"].is_null().sum()),\n        }\n\n    head = panel_lf.select(pl.len().alias(\"rows\"), pl.col(\"d_channel_key\").n_unique().alias(\"channels\"),\n                           pl.col(\"d_cutoff_date\").min().alias(\"dmin\"), pl.col(\"d_cutoff_date\").max().alias(\"dmax\")).collect()\n    out: dict[str, Any] = {\"rows\": int(head[\"rows\"][0]), \"channels\": int(head[\"channels\"][0]),\n                           \"date_min\": str(head[\"dmin\"][0]), \"date_max\": str(head[\"dmax\"][0]),\n                           \"link_episodes\": ep_stats(\"link\"), \"power_episodes\": ep_stats(\"power\"), \"targets\": {}}\n    for t in TARGETS:\n        r = f\"reason_{t[len('target_'):]}\"\n        g = panel_lf.group_by(r).agg(pl.len()).collect()\n        by_year = panel_lf.group_by(\"d_year\").agg((pl.col(t) == 1).sum().alias(\"pos\"), (pl.col(t) == 0).sum().alias(\"neg\"),\n                                                  pl.col(t).null_count().alias(\"null\")).sort(\"d_year\").collect()\n        reasons = {k: int(v) for k, v in g.iter_rows()}\n        out[\"targets\"][t] = {\n            \"positive\": reasons.get(\"positive\", 0),\n            \"negative\": reasons.get(\"negative\", 0) + reasons.get(\"negative_transient_only\", 0),\n            \"null_reasons\": {k: v for k, v in reasons.items() if k not in (\"positive\", \"negative\", \"negative_transient_only\")},\n            \"negative_transient_only\": reasons.get(\"negative_transient_only\", 0),\n            \"by_year\": {str(y): {\"pos\": int(p), \"neg\": int(n), \"null\": int(nu)} for y, p, n, nu in by_year.iter_rows()},\n        }\n    return out\n\n\n# ---------------------------------------------------------------------------\n# 7. Поток по годам (для Kaggle): сырой журнал → части → панель\n# ---------------------------------------------------------------------------\ndef read_journal(pl: Any, path: Path) -> Any:\n    schema = {c: pl.String for c in EVENT_COLUMNS}\n    return pl.read_csv(path, schema_overrides=schema, infer_schema_length=0, encoding=\"utf8-lossy\",\n                       columns=EVENT_COLUMNS).filter(pl.col(\"ид_события\") != \"ид_события\")\n\n\ndef read_catalogue(pl: Any, path: Path) -> Any:\n    cat = pl.read_csv(path, infer_schema_length=0, encoding=\"utf8-lossy\")\n    return cat.select([c for c in CATALOGUE_COLUMNS if c in cat.columns]).unique(\"ид_канала_данных\", keep=\"last\")\n\n\ndef process_year(pl: Any, raw: Any, tax: dict[str, Any], catalogue: Any, gas_carry: Any) -> dict[str, Any]:\n    ev = classify_events(pl, raw, tax, catalogue)\n    sec = second_table(pl, ev, tax)\n    daily = daily_aggregates(pl, ev, sec, tax)\n    gas_daily, gas_carry = gas_crossings(pl, ev, gas_carry, tax[\"numeric\"][\"gas_threshold_percent\"])\n    meta = ev.group_by(\"ch\").agg(pl.col(\"тип_датчика\").drop_nulls().last(), pl.col(\"тип_инж_системы\").drop_nulls().last(),\n                                 pl.col(\"ид_объект\").drop_nulls().last())\n    class_counts = {k: int(v) for k, v in ev.group_by(\"cls\").len().iter_rows()}\n    streams = tax[\"stream_mapping\"][\"streams\"]\n    mapped = {c for st in streams.values() for key in (\"fault\", \"ok\", \"unknown\") for c in st[key]}\n    stream_audit = {}\n    for name in streams:\n        g = sec.group_by(f\"{name}_cls\", f\"{name}_a_cause\").len()\n        stream_audit[name] = {f\"{st or 'none'}{'_' + cause if cause else ''}\": int(n) for st, cause, n in g.iter_rows()}\n    stream_audit[\"events_in_no_stream_by_class\"] = {k: v for k, v in class_counts.items() if k not in mapped}\n    # «Неопределен»: сколько каналов и какие классы соседствуют (агрегаты, порядок внутри секунды не используется)\n    evs = ev.select(\"ch\", \"t\", \"cls\").sort(\"ch\", \"t\")\n    evs = evs.with_columns(pl.col(\"cls\").shift(1).over(\"ch\").alias(\"prev\"), pl.col(\"cls\").shift(-1).over(\"ch\").alias(\"next\"))\n    svc = evs.filter(pl.col(\"cls\") == \"service\")\n    service_audit = {\"events\": svc.height, \"channels\": svc[\"ch\"].n_unique(),\n                     \"prev_class\": {str(k): int(v) for k, v in svc.group_by(\"prev\").len().sort(\"len\", descending=True).head(10).iter_rows()},\n                     \"next_class\": {str(k): int(v) for k, v in svc.group_by(\"next\").len().sort(\"len\", descending=True).head(10).iter_rows()}}\n    unknown_values = {k: int(v) for k, v in ev.filter(pl.col(\"cls\") == \"unknown_value\").group_by(\"v\").len()\n                      .sort(\"len\", descending=True).head(30).iter_rows()}\n    return {\n        \"daily\": daily, \"link_cp\": change_points(pl, sec, \"link_cls\"), \"power_cp\": change_points(pl, sec, \"power_cls\"),\n        \"gas_daily\": gas_daily, \"gas_carry\": gas_carry, \"meta\": meta, \"t_max\": ev[\"t\"].max(),\n        \"audit\": {\"events\": ev.height, \"channels\": ev[\"ch\"].n_unique(), \"seconds\": sec.height,\n                  \"multi_value_seconds\": int((sec[\"n_values\"] > 1).sum()),\n                  \"link_fault_seconds\": int((sec[\"link_cls\"] == \"F\").sum()),\n                  \"link_fault_seconds_with_other_value\": int(((sec[\"has_fault\"]) & (sec[\"n_values\"] > 1)).sum()),\n                  \"link_ambiguous_seconds\": int((sec[\"link_cls\"] == \"A\").sum()),\n                  \"class_counts\": class_counts, \"unknown_values_top30\": unknown_values,\n                  \"stream_mapping\": stream_audit, \"service_value\": service_audit},\n    }\n\n\ndef empty_gas_carry(pl: Any) -> Any:\n    return pl.DataFrame(schema={\"ch\": pl.String, \"t\": pl.Datetime(\"us\"), \"mn\": pl.Float64, \"mx\": pl.Float64})\n\n\ndef journal_max_time(pl: Any, path: Path) -> datetime:\n    \"\"\"Лёгкий предпроход: последний момент в файле (для общей границы данных).\"\"\"\n    t = pl.concat_str([\"дата\", \"время\"], separator=\" \").str.strptime(pl.Datetime(\"us\"), \"%Y-%m-%d %H:%M:%S\", strict=False)\n    return _scan_source(pl, path).select(t.max()).collect().item()\n\n\ndef _scan_source(pl: Any, path: Path) -> Any:\n    path = Path(path)\n    if path.suffix == \".parquet\":\n        return pl.scan_parquet(path)\n    schema = {c: pl.String for c in EVENT_COLUMNS}\n    return pl.scan_csv(path, schema_overrides=schema, infer_schema_length=0, encoding=\"utf8-lossy\")\n\n\ndef scan_bucket(pl: Any, path: Path, bucket: int, n_buckets: int, channel_filter: Any = None) -> Any:\n    lf = _scan_source(pl, path).select(EVENT_COLUMNS).filter(pl.col(\"ид_события\") != \"ид_события\")\n    if n_buckets > 1:\n        lf = lf.filter(pl.col(\"ид_канала_данных\").hash(seed=20260923) % n_buckets == bucket)\n    if channel_filter is not None:\n        lf = lf.filter(channel_filter)\n    return lf.collect()\n\n\ndef _merge_audits(parts: list[dict[str, Any]]) -> dict[str, Any]:\n    out: dict[str, Any] = {}\n    for a in parts:\n        for k, v in a.items():\n            if isinstance(v, bool) or v is None:\n                out.setdefault(k, v)\n            elif isinstance(v, (int, float)):\n                out[k] = out.get(k, 0) + v\n            elif isinstance(v, dict):\n                out[k] = _merge_audits([out.get(k, {}), v])\n            else:\n                out.setdefault(k, v)\n    return out\n\n\ndef build_event_panel(pl: Any, journal_paths: list[Path], catalogue_path: Path, taxonomy_path: Path,\n                      config: dict[str, Any] | None = None, log: Any = print, n_buckets: int = 1,\n                      out_dir: Path | None = None, channel_filter: Any = None,\n                      data_end: datetime | None = None) -> tuple[Any, dict[str, Any]]:\n    \"\"\"Собирает панель по группам каналов (каналы независимы). При out_dir пишет части parquet и\n    возвращает (None, audit); иначе возвращает панель в памяти.\"\"\"\n    cfg = {**DEFAULT_CONFIG, **(config or {})}\n    tax = load_taxonomy(taxonomy_path)\n    cfg[\"gas_threshold_percent\"] = tax[\"numeric\"][\"gas_threshold_percent\"]\n    if not cfg[\"service_as_unknown\"]:\n        link = tax[\"stream_mapping\"][\"streams\"][\"link\"]\n        link[\"unknown\"] = [c for c in link[\"unknown\"] if c != \"service\"]\n    if cfg[\"power_ok_from_function\"]:\n        power = tax[\"stream_mapping\"][\"streams\"][\"power\"]\n        power[\"ok\"] = list(dict.fromkeys(power[\"ok\"] + [\"normal\", \"event_alarm\", \"numeric\"]))\n    cfg[\"stream_mapping_version\"] = tax[\"stream_mapping\"][\"version\"]\n    catalogue = read_catalogue(pl, catalogue_path)\n    # data_end задаётся явно на runtime (конец дня D): события после D в сборку не попадают (см. channel_filter)\n    t_max = data_end if data_end is not None else max(journal_max_time(pl, p) for p in journal_paths)\n    panels, part_paths, per_file_parts, ep_parts = [], [], [], []\n    for b in range(n_buckets):\n        carry = empty_gas_carry(pl)\n        parts: dict[str, list[Any]] = {\"daily\": [], \"link_cp\": [], \"power_cp\": [], \"gas_daily\": [], \"meta\": []}\n        per_file = {}\n        for path in journal_paths:\n            raw = scan_bucket(pl, Path(path), b, n_buckets, channel_filter)\n            if raw.is_empty():\n                continue\n            res = process_year(pl, raw, tax, catalogue, carry)\n            carry = res[\"gas_carry\"]\n            for k in parts:\n                parts[k].append(res[k])\n            per_file[Path(path).name] = res[\"audit\"]\n            del raw, res\n        if not parts[\"daily\"]:\n            continue\n        daily = pl.concat(parts[\"daily\"], how=\"vertical_relaxed\")\n        dup = daily.height - daily.select(\"ch\", \"day\").unique().height\n        if dup:\n            raise ValueError(f\"дневные агрегаты пересекаются между файлами: {dup} строк\")\n        link_cp = rle(pl, pl.concat(parts[\"link_cp\"], how=\"vertical\"))\n        power_cp = rle(pl, pl.concat(parts[\"power_cp\"], how=\"vertical\"))\n        gas_daily = pl.concat(parts[\"gas_daily\"], how=\"vertical\").group_by(\"ch\", \"day\").agg(pl.all().sum())\n        meta = pl.concat(parts[\"meta\"], how=\"vertical_relaxed\").group_by(\"ch\").agg(pl.all().drop_nulls().last())\n        panel, audit = build_panel_from_parts(pl, daily, link_cp, power_cp, gas_daily, meta, t_max, cfg)\n        per_file_parts.append({\"per_file\": per_file})\n        ep_parts.append(audit.pop(\"_episode_durations\"))\n        if out_dir is not None:\n            path = Path(out_dir) / f\"event_panel_v3_part{b:02d}.parquet\"\n            panel.write_parquet(path, compression=\"zstd\")\n            part_paths.append(path)\n            del panel\n        else:\n            panels.append(panel)\n        log(f\"группа каналов {b + 1}/{n_buckets}: строк {audit['rows']:,}, каналов {audit['channels']:,}\")\n    if out_dir is not None:\n        full = pl.scan_parquet(part_paths)\n    else:\n        full = pl.concat(panels, how=\"vertical_relaxed\").lazy()\n    audit = final_audit(pl, full, pl.concat(ep_parts, how=\"vertical\"))\n    audit[\"feature_columns\"] = [c for c in full.collect_schema().names() if c.startswith(\"f_\")] + BASE_FEATURES\n    audit[\"target_columns\"] = TARGETS\n    audit.update({\"schema_version\": SCHEMA_VERSION, \"config\": cfg, \"taxonomy_version\": tax[\"version\"],\n                  \"taxonomy_sha256\": tax[\"_sha256\"], \"data_end\": str(t_max), \"n_buckets\": n_buckets,\n                  \"per_file\": _merge_audits(per_file_parts)[\"per_file\"] if per_file_parts else {},\n                  \"part_files\": [p.name for p in part_paths]})\n    return (None if out_dir is not None else full.collect()), audit\n\n\ndef csv_to_parquet(pl: Any, csv_path: Path, out_path: Path) -> dict[str, Any]:\n    \"\"\"Сырой CSV → parquet только нужных колонок (все строки как строки, без интерпретации).\"\"\"\n    schema = {c: pl.String for c in EVENT_COLUMNS}\n    lf = pl.scan_csv(csv_path, schema_overrides=schema, infer_schema_length=0, encoding=\"utf8-lossy\").select(EVENT_COLUMNS)\n    lf.sink_parquet(out_path, compression=\"zstd\")\n    rows = pl.scan_parquet(out_path).select(pl.len()).collect().item()\n    return {\"rows\": int(rows), \"parquet_bytes\": Path(out_path).stat().st_size}\n\n\n# ---------------------------------------------------------------------------\n# 8. Самопроверки на синтетике (для notebook)\n# ---------------------------------------------------------------------------\ndef run_event_panel_self_tests(taxonomy_path: Path, work_dir: Path) -> dict[str, bool]:\n    import polars as pl\n    import random\n\n    work_dir = Path(work_dir)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    t0 = datetime(2024, 3, 1)\n\n    def events(rows):\n        return pl.DataFrame({\"ид_события\": [str(i) for i in range(len(rows))],\n                             \"ид_канала_данных\": [r[0] for r in rows],\n                             \"дата\": [r[1].strftime(\"%Y-%m-%d\") for r in rows],\n                             \"время\": [r[1].strftime(\"%H:%M:%S\") for r in rows],\n                             \"тревожное\": [\"f\"] * len(rows), \"значение_датчика\": [r[2] for r in rows]})\n\n    def catalogue(chs, typ=\"Состояние насоса\"):\n        return pl.DataFrame({\"ид_канала_данных\": chs, \"тип_инж_системы\": [\"x\"] * len(chs),\n                             \"тип_датчика\": [typ] * len(chs), \"ид_объект\": [\"1\"] * len(chs)})\n\n    def build(name, rows, chs, **kw):\n        d = work_dir / name\n        d.mkdir(exist_ok=True)\n        events(rows).write_csv(d / \"j.csv\")\n        catalogue(chs).write_csv(d / \"c.csv\")\n        return build_event_panel(pl, [d / \"j.csv\"], d / \"c.csv\", taxonomy_path, log=lambda *a: None, **kw)\n\n    def row(panel, ch, day):\n        r = panel.filter((pl.col(\"d_channel_key\") == pseudo_key(ch)) & (pl.col(\"d_cutoff_date\") == (t0 + timedelta(days=day)).date()))\n        return r.row(0, named=True)\n\n    report: dict[str, bool] = {}\n    hb = [(\"A\", t0 + timedelta(days=d, hours=12), \"Норма\") for d in range(12)]\n    p, _ = build(\"onset\", hb + [(\"A\", t0 + timedelta(days=5, hours=13), \"Неисправен\"), (\"A\", t0 + timedelta(days=5, hours=14), \"Норма\")], [\"A\"])\n    report[\"strict_onset_in_window_is_positive\"] = row(p, \"A\", 3)[\"target_t2a_link_onset\"] == 1\n    report[\"onset_in_d1_is_competing_not_filtered\"] = row(p, \"A\", 4)[\"target_t2a_link_onset\"] == 0 and row(p, \"A\", 4)[\"competing_link_onset_d1\"]\n    t = t0 + timedelta(days=5, hours=13)\n    p, _ = build(\"amb\", hb + [(\"A\", t, \"Неисправен\"), (\"A\", t, \"Норма\")], [\"A\"])\n    r = row(p, \"A\", 3)  # конфликт в одной секунде → не onset и не 0, а null(ambiguous_onset)\n    report[\"same_second_conflict_is_null\"] = r[\"target_t2a_link_onset\"] is None and r[\"reason_t2a_link_onset\"] == \"ambiguous_onset\"\n    rnd = random.Random(1)\n    vals = [\"Норма\", \"Неисправен\", \"Обесточен\", \"Есть питание\", \"Выключен\", \"Неопределен\", \"0,05\", \"1,3\", \"-127\"]\n    rows = [(f\"C{c}\", t0 + timedelta(days=d, seconds=rnd.randint(0, 86399)), rnd.choice(vals))\n            for c in range(8) for d in range(30) for _ in range(rnd.randint(0, 5))]\n    chs = [f\"C{c}\" for c in range(8)]\n    base, _ = build(\"rand1\", rows, chs)\n    out = work_dir / \"rand_parts\"\n    out.mkdir(exist_ok=True)\n    build(\"rand2\", rows, chs, n_buckets=3, out_dir=out)\n    parts = pl.read_parquet(sorted(out.glob(\"*.parquet\")))\n    key = [\"d_channel_key\", \"d_cutoff_date\"]\n    report[\"channel_buckets_do_not_change_panel\"] = base.sort(key).equals(parts.sort(key))\n    report[\"no_raw_identifiers_in_panel\"] = not any(c in base.write_csv() for c in (\"C1,\", \"C2,\"))\n    return report\n\n\ndef label_changes(pl: Any, main: Any, variant: Any, targets: list[str] | None = None) -> dict[str, dict[str, int]]:\n    \"\"\"Сколько меток меняется между основной и sensitivity-версией (только агрегаты, по ключу строки).\"\"\"\n    k = [\"d_channel_key\", \"d_cutoff_date\"]\n    out: dict[str, dict[str, int]] = {}\n    for t in targets or TARGETS:\n        j = main.select(k + [t]).join(variant.select(k + [t]), on=k, how=\"full\", suffix=\"_v\", coalesce=True)\n        a = pl.col(t).cast(pl.String).fill_null(\"null\")\n        b = pl.col(f\"{t}_v\").cast(pl.String).fill_null(\"null\")\n        g = j.with_columns(a.alias(\"a\"), b.alias(\"b\")).filter(pl.col(\"a\") != pl.col(\"b\")).group_by(\"a\", \"b\").len()\n        out[t] = {f\"{x}->{y}\": int(n) for x, y, n in g.sort(\"a\", \"b\").iter_rows()}\n    return out\n",
"event_panel_v3_kaggle.py": "\"\"\"Kaggle-обвязка notebook 14: сырые журналы → событийная панель v3 (части parquet) + manifest/аудит/сводка.\n\nШаги:\n1. поиск входов в /kaggle/input: ext-journal-YYYY.{7z,csv,parquet} и справочник_каналов_датчиков.csv\n   (имена сравниваются после Unicode-NFC: загрузка с macOS даёт NFD-кириллицу);\n2. распаковка 7z по одному архиву → parquet только нужных колонок (все значения строками) → CSV удаляется;\n3. перераскладка по календарным годам события (файл года может содержать события соседнего года;\n   иначе дневные агрегаты пересеклись бы между файлами) и удаление точных дублей строк (счётчик в аудите);\n4. сборка панели по группам каналов (каналы независимы) → event_panel_v3/event_panel_v3_partNN.parquet;\n5. manifest с SHA-256 частей и входов, аудит и сводка — только агрегаты, без идентификаторов.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport unicodedata\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\ntry:\n    from event_panel_v3 import (EVENT_COLUMNS, SCHEMA_VERSION, SENSITIVITY_VARIANTS, TARGETS, build_event_panel,\n                                csv_to_parquet, label_changes)\nexcept ImportError:  # в notebook имена уже определены ячейкой выше\n    pass\n\nJOURNAL_RE = re.compile(r\"^ext-journal-(\\d{4})\\.(7z|csv|parquet)$\")\nCATALOGUE_NAME = \"справочник_каналов_датчиков.csv\"\nSOURCE_PRIORITY = {\"parquet\": 0, \"csv\": 1, \"7z\": 2}\nOUTPUT_FILES = (\"panel_manifest_v3.json\", \"audit_event_panel_v3.json\", \"summary_event_panel_v3_ru.md\")\n\n\ndef make_config_v3(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    if mode not in (\"FULL\", \"SMOKE\"):\n        raise ValueError(\"MODE должен быть FULL или SMOKE\")\n    cfg: dict[str, Any] = {\n        \"mode\": mode,\n        \"input_dir\": os.environ.get(\"LDT_KAGGLE_INPUT_DIR\", \"/kaggle/input\"),\n        \"output_dir\": os.environ.get(\"LDT_OUTPUT_DIR\", \"/kaggle/working\"),\n        \"stage_dir\": os.environ.get(\"LDT_STAGE_DIR\", \"/tmp/ldt_event_panel_v3_stage\"),\n        \"taxonomy_path\": None,\n        \"n_buckets\": 4,\n        \"panel\": {\"tau_minutes\": 5},\n        \"smoke_years\": [2019, 2020],\n        \"smoke_channel_share\": 20,   # SMOKE: ≈1/20 каналов\n        \"sensitivity_variants\": [\"service_neutral\", \"power_ok_from_function\"],  # отдельные сборки, в выход — только агрегаты\n        \"keep_stage\": False,\n    }\n    cfg.update(overrides or {})\n    return cfg\n\n\ndef _nfc(name: str) -> str:\n    return unicodedata.normalize(\"NFC\", name)\n\n\ndef file_sha256(path: Path, chunk: int = 1 << 22) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        while block := f.read(chunk):\n            h.update(block)\n    return h.hexdigest()\n\n\nCATALOGUE_REQUIRED_COLUMNS = {\"ид_канала_данных\", \"тип_датчика\"}\nYEAR_IN_PATH_RE = re.compile(r\"ext-journal-(\\d{4})\")\n\n\ndef _csv_header(path: Path) -> set[str]:\n    \"\"\"Имена колонок первой строки CSV (для файлов, которые Kaggle переименовал: «справочник…» → «__.csv»).\"\"\"\n    try:\n        with open(path, \"r\", encoding=\"utf-8-sig\", errors=\"replace\") as f:\n            line = f.readline()\n    except OSError:\n        return set()\n    return {c.strip().strip('\"') for c in line.rstrip(\"\\r\\n\").split(\",\")}\n\n\ndef discover_sources(input_dir: Path) -> dict[str, Any]:\n    \"\"\"Один источник на год (parquet > csv > 7z) и один справочник каналов.\n\n    Сначала по имени файла; для CSV с непонятным именем — по заголовку. Kaggle при загрузке заменяет кириллицу\n    в имени файла на «_» (справочник становится «__.csv») и распаковывает архивы в папки ext-journal-YYYY/.\"\"\"\n    by_year: dict[int, list[tuple[int, Path]]] = {}\n    catalogues: list[Path] = []\n    for p in sorted(Path(input_dir).rglob(\"*\")):\n        if not p.is_file():\n            continue\n        name = _nfc(p.name)\n        m = JOURNAL_RE.match(name)\n        if m:\n            by_year.setdefault(int(m.group(1)), []).append((SOURCE_PRIORITY[m.group(2)], p))\n            continue\n        if name == CATALOGUE_NAME:\n            catalogues.insert(0, p)\n            continue\n        if p.suffix.lower() != \".csv\":\n            continue\n        header = _csv_header(p)\n        if set(EVENT_COLUMNS) <= header:\n            ym = YEAR_IN_PATH_RE.search(_nfc(str(p)))\n            if ym:  # журнал с неожиданным именем файла, год — из пути (папка ext-journal-YYYY)\n                by_year.setdefault(int(ym.group(1)), []).append((SOURCE_PRIORITY[\"csv\"] + 5, p))\n        elif CATALOGUE_REQUIRED_COLUMNS <= header and \"значение_датчика\" not in header:\n            catalogues.append(p)\n    journals = {y: sorted(v)[0][1] for y, v in sorted(by_year.items())}\n    years = sorted(journals)\n    missing = [y for y in range(years[0], years[-1] + 1) if y not in journals] if years else []\n    return {\"journals\": journals, \"ignored_duplicates\": sum(len(v) - 1 for v in by_year.values()),\n            \"catalogue\": catalogues[0] if catalogues else None, \"catalogue_candidates\": len(catalogues),\n            \"missing_years\": missing}\n\n\ndef describe_sources(src: dict[str, Any]) -> list[str]:\n    \"\"\"Строки для preflight-ячейки: что найдено и чего не хватает (без содержимого файлов).\"\"\"\n    lines = [f\"журналы: {', '.join(str(y) for y in src['journals']) or 'не найдены'}\"]\n    for y, p in src[\"journals\"].items():\n        lines.append(f\"  {y}: {p.parent.name}/{p.name} ({p.stat().st_size / 2**30:.2f} ГБ)\")\n    cat = src[\"catalogue\"]\n    lines.append(f\"справочник каналов: {cat.name if cat else 'НЕ НАЙДЕН'}\"\n                 + (f\" (кандидатов: {src['catalogue_candidates']})\" if src[\"catalogue_candidates\"] > 1 else \"\"))\n    if src[\"missing_years\"]:\n        lines.append(\"ВНИМАНИЕ: нет журналов за \" + \", \".join(map(str, src[\"missing_years\"]))\n                     + \" — фолды ноутбука 15 обучаются на 2019–2020 и 2022+; без пропущенного года train неполный\")\n    return lines\n\n\ndef extract_7z(archive: Path, dest: Path, log: Any = print) -> list[Path]:\n    dest.mkdir(parents=True, exist_ok=True)\n    exe = next((shutil.which(x) for x in (\"7z\", \"7za\", \"7zr\") if shutil.which(x)), None)\n    if exe:\n        subprocess.run([exe, \"x\", \"-y\", f\"-o{dest}\", str(archive)], check=True,\n                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)\n    else:\n        try:\n            import py7zr  # type: ignore\n        except ImportError:\n            log(\"7z не найден — ставлю py7zr (нужен Internet в Settings ноутбука)\")\n            try:\n                subprocess.check_call([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"py7zr\"])\n                import py7zr  # type: ignore\n            except Exception as exc:  # noqa: BLE001\n                raise RuntimeError(\"Нет распаковщика 7z: включите Internet в Settings или загрузите в dataset \"\n                                   \"распакованные ext-journal-YYYY.csv\") from exc\n        with py7zr.SevenZipFile(archive, \"r\") as z:\n            z.extractall(dest)\n    return sorted(p for p in dest.rglob(\"*\") if p.is_file() and p.suffix.lower() == \".csv\")\n\n\ndef stage_journals(pl: Any, journals: dict[int, Path], stage_dir: Path, years: list[int] | None,\n                   log: Any = print) -> tuple[list[Path], dict[str, Any]]:\n    \"\"\"Источники → parquet → файлы по календарному году события. Возвращает (файлы годов, аудит).\"\"\"\n    raw_dir, tmp_dir, year_dir = stage_dir / \"raw_parquet\", stage_dir / \"extract\", stage_dir / \"by_year\"\n    for d in (raw_dir, year_dir):\n        shutil.rmtree(d, ignore_errors=True)\n        d.mkdir(parents=True)\n    audit: dict[str, Any] = {\"sources\": [], \"years\": {}}\n    raw_files: list[Path] = []\n    for year, src in journals.items():\n        if years is not None and year not in years:\n            continue\n        t0 = time.time()\n        entry = {\"name\": _nfc(src.name), \"bytes\": src.stat().st_size, \"sha256\": file_sha256(src), \"rows\": 0}\n        if src.suffix == \".parquet\":\n            raw_files.append(src)\n            entry[\"rows\"] = int(pl.scan_parquet(src).select(pl.len()).collect().item())\n        else:\n            csvs = [src] if src.suffix == \".csv\" else extract_7z(src, tmp_dir, log)\n            if not csvs:\n                raise RuntimeError(f\"в архиве {entry['name']} нет CSV\")\n            for i, csv in enumerate(csvs):\n                out = raw_dir / f\"{src.stem}_{i}.parquet\"\n                entry[\"rows\"] += csv_to_parquet(pl, csv, out)[\"rows\"]\n                raw_files.append(out)\n            shutil.rmtree(tmp_dir, ignore_errors=True)\n        audit[\"sources\"].append(entry)\n        log(f\"{entry['name']}: {entry['rows']:,} строк → parquet за {time.time() - t0:.0f} с\")\n    if not raw_files:\n        return [], audit\n    prefix = pl.col(\"дата\").str.slice(0, 4)\n    is_header = pl.col(\"ид_события\") == \"ид_события\"\n    # построчный баланс по каждому источнику: повторённые строки заголовка и строки без года тоже учитываются явно\n    per_file = (pl.concat([pl.scan_parquet(f).select(pl.lit(f.name).alias(\"_file\"), is_header.alias(\"_hdr\"),\n                                                     prefix.alias(\"y\")) for f in raw_files])\n                .group_by(\"_file\").agg(pl.col(\"_hdr\").sum().alias(\"header_rows\"),\n                                       (~pl.col(\"_hdr\") & ~pl.col(\"y\").fill_null(\"\").str.contains(r\"^\\d{4}$\"))\n                                       .sum().alias(\"unparseable_year_rows\")).collect())\n    audit[\"raw_files\"] = {name: {\"header_rows\": int(h), \"unparseable_year_rows\": int(u)}\n                          for name, h, u in per_file.iter_rows()}\n    counts = (pl.scan_parquet(raw_files).filter(~is_header)\n              .group_by(prefix.alias(\"y\")).agg(pl.len()).collect())\n    bad = int(counts.filter(~pl.col(\"y\").str.contains(r\"^\\d{4}$\") | pl.col(\"y\").is_null())[\"len\"].sum())\n    audit[\"rows_with_unparseable_year_excluded\"] = bad\n    audit[\"header_rows_excluded\"] = int(per_file[\"header_rows\"].sum())\n    year_files = []\n    for y in sorted(counts.filter(pl.col(\"y\").str.contains(r\"^\\d{4}$\"))[\"y\"].to_list()):\n        if years is not None and int(y) not in years:\n            audit[\"years\"][y] = {\"skipped_outside_mode_years\": int(counts.filter(pl.col(\"y\") == y)[\"len\"][0])}\n            continue\n        frames = []\n        for f in raw_files:\n            part = pl.scan_parquet(f).select(EVENT_COLUMNS).filter((prefix == y) & (pl.col(\"ид_события\") != \"ид_события\")).collect()\n            if part.height:\n                frames.append((f, part))\n        rows = pl.concat([p for _, p in frames], how=\"vertical\")\n        n_in = rows.height\n        rows = rows.unique(maintain_order=True)\n        # диагностика дублей: одинаковый ид_события при разном содержимом (такие строки НЕ удаляются)\n        same_id_diff = rows.height - rows.select(pl.col(\"ид_события\").n_unique()).item()\n        from_other = sum(p.height for f, p in frames if f\"-{y}\" not in f.name)\n        out = year_dir / f\"journal_{y}.parquet\"\n        rows.write_parquet(out, compression=\"zstd\")\n        audit[\"years\"][y] = {\"rows_in\": n_in, \"rows\": rows.height, \"exact_duplicate_rows_dropped\": n_in - rows.height,\n                             \"same_event_id_different_content_kept\": int(same_id_diff),\n                             \"rows_from_other_year_files\": from_other}\n        year_files.append(out)\n        del rows, frames\n    src_rows = sum(e[\"rows\"] for e in audit[\"sources\"])\n    staged = sum(v.get(\"rows\", 0) for v in audit[\"years\"].values())\n    dups = sum(v.get(\"exact_duplicate_rows_dropped\", 0) for v in audit[\"years\"].values())\n    skipped = sum(v.get(\"skipped_outside_mode_years\", 0) for v in audit[\"years\"].values())\n    audit[\"row_balance\"] = {\"source_rows\": src_rows, \"staged_rows\": staged, \"exact_duplicates_dropped\": dups,\n                            \"header_rows_excluded\": audit[\"header_rows_excluded\"],\n                            \"unparseable_year_rows_excluded\": bad, \"skipped_outside_mode_years\": skipped,\n                            \"unexplained\": src_rows - staged - dups - audit[\"header_rows_excluded\"] - bad - skipped}\n    if not os.environ.get(\"LDT_KEEP_RAW_STAGE\"):\n        shutil.rmtree(raw_dir, ignore_errors=True)\n    return year_files, audit\n\n\ndef _truncate_unknown(audit: dict[str, Any]) -> None:\n    \"\"\"Неизвестные значения — только короткие строки (не выгружать длинный свободный текст).\"\"\"\n    for fa in audit.get(\"per_file\", {}).values():\n        top = fa.get(\"unknown_values_top30\")\n        if top:\n            fa[\"unknown_values_top30\"] = {str(k)[:40]: v for k, v in top.items()}\n\n\ndef summary_markdown(manifest: dict[str, Any], audit: dict[str, Any]) -> str:\n    L = [f\"# Событийная панель v3 — сводка ({manifest['mode']})\", \"\"]\n    if manifest[\"mode\"] == \"SMOKE\":\n        L += [\"**SMOKE:** ≈1/{} каналов, годы {}. Цифры не сравнимы с FULL.\".format(\n            manifest[\"config\"][\"smoke_channel_share\"], manifest[\"config\"][\"smoke_years\"]), \"\"]\n    L += [f\"Строк (канал × сутки D): **{audit['rows']:,}**, каналов: {audit['channels']:,}, \"\n          f\"D от {audit['date_min']} до {audit['date_max']}; граница данных {audit['data_end']}.\", \"\",\n          f\"Время сборки: {manifest['runtime_s']:.0f} с; частей: {len(manifest['parts'])}.\", \"\",\n          \"## Цели\", \"\",\n          \"| цель | 1 | 0 | null | доля 1 среди размеченных | главные причины null |\", \"|---|---|---|---|---|---|\"]\n    for t, s in audit[\"targets\"].items():\n        null = sum(s[\"null_reasons\"].values())\n        lab = s[\"positive\"] + s[\"negative\"]\n        top = \", \".join(f\"{k} {v:,}\" for k, v in sorted(s[\"null_reasons\"].items(), key=lambda kv: -kv[1])[:3])\n        L.append(f\"| `{t}` | {s['positive']:,} | {s['negative']:,} | {null:,} | \"\n                 f\"{(s['positive'] / lab if lab else float('nan')):.4f} | {top} |\")\n    L += [\"\", \"## Эпизоды (секундный уровень)\", \"\", \"| вид | эпизодов | strict onset | ambiguous | медиана длит., с | ≥ τ | < τ | цензура |\",\n          \"|---|---|---|---|---|---|---|---|\"]\n    for kind in (\"link_episodes\", \"power_episodes\"):\n        e = audit[kind]\n        med = (e[\"exact_duration_s_quantiles\"] or {}).get(\"0.5\")\n        L.append(f\"| {kind.split('_')[0]} | {e['episodes']:,} | {e['strict_onsets']:,} | {e['ambiguous_onsets']:,} | \"\n                 f\"{med if med is not None else '—'} | {e['sustained_1']:,} | {e['sustained_0']:,} | {e['sustained_null']:,} |\")\n    cls: dict[str, int] = {}\n    ev = 0\n    for fa in audit[\"per_file\"].values():\n        ev += fa.get(\"events\", 0)\n        for k, v in fa.get(\"class_counts\", {}).items():\n            cls[k] = cls.get(k, 0) + v\n    L += [\"\", \"## Классы значений\", \"\", \"| класс | событий | доля |\", \"|---|---|---|\"]\n    for k, v in sorted(cls.items(), key=lambda kv: -kv[1]):\n        L.append(f\"| {k} | {v:,} | {v / max(ev, 1):.4f} |\")\n    unk = cls.get(\"unknown_value\", 0) / max(ev, 1)\n    L += [\"\", \"## Предупреждения\", \"\"]\n    warn = []\n    if unk > 0.01:\n        warn.append(f\"- доля неизвестных значений {unk:.3f} > 1 % — дополнить таксономию до обучения\")\n    t2a = audit[\"targets\"][\"target_t2a_link_onset\"]\n    unobs = t2a[\"null_reasons\"].get(\"unobserved_window\", 0)\n    if unobs > t2a[\"positive\"] + t2a[\"negative\"]:\n        warn.append(f\"- T2a: окон без событий ({unobs:,}) больше, чем размеченных — событийные каналы молчат; \"\n                    \"coverage показывать отдельно\")\n    stage = manifest[\"stage\"]\n    moved = sum(v.get(\"rows_from_other_year_files\", 0) for v in stage[\"years\"].values())\n    dups = sum(v.get(\"exact_duplicate_rows_dropped\", 0) for v in stage[\"years\"].values())\n    rb = stage.get(\"row_balance\", {})\n    if rb:\n        L_rb = (f\"- баланс строк: источники {rb['source_rows']:,} = в панель {rb['staged_rows']:,} + точные дубли \"\n                f\"{rb['exact_duplicates_dropped']:,} + строки заголовка {rb['header_rows_excluded']:,} + без года \"\n                f\"{rb['unparseable_year_rows_excluded']:,} + вне режима {rb['skipped_outside_mode_years']:,}; \"\n                f\"необъяснённых: **{rb['unexplained']:,}**\")\n        warn.append(L_rb)\n    if manifest.get(\"missing_years\"):\n        warn.append(\"- **нет журналов за \" + \", \".join(map(str, manifest[\"missing_years\"])) + \"** — панель неполная, \"\n                    \"train фолдов ноутбука 15 без этого года\")\n    if moved:\n        warn.append(f\"- {moved:,} строк лежали в файле другого года — перенесены по дате события\")\n    if dups:\n        warn.append(f\"- удалено точных дублей строк: {dups:,}\")\n    L += warn or [\"- нет\"]\n    sm = {}\n    for fa in audit[\"per_file\"].values():\n        for stream, counts in fa.get(\"stream_mapping\", {}).items():\n            if stream == \"events_in_no_stream_by_class\":\n                continue\n            for k, v in counts.items():\n                sm.setdefault(stream, {}).setdefault(k, 0)\n                sm[stream][k] += v\n    if sm:\n        L += [\"\", f\"## Потоки (stream mapping {manifest['panel_config'].get('stream_mapping_version', '?')}): секунды по состоянию\", \"\",\n              \"| поток | F | O | A (конфликт) | A (неизвестно) | без сведений |\", \"|---|---|---|---|---|---|\"]\n        for stream, c in sm.items():\n            L.append(f\"| {stream} | {c.get('F', 0):,} | {c.get('O', 0):,} | {c.get('A_conflict', 0):,} | \"\n                     f\"{c.get('A_unknown', 0):,} | {c.get('none', 0):,} |\")\n    if audit.get(\"sensitivity\"):\n        L += [\"\", \"## Sensitivity-версии разметки (заранее объявлены; выбор не по validation-метрикам)\", \"\"]\n        for name, sv in audit[\"sensitivity\"].items():\n            L.append(f\"**{name}** ({sv['config']}): изменений меток —\")\n            for t, ch in sv[\"label_changes\"].items():\n                if ch:\n                    L.append(f\"- `{t}`: \" + \", \".join(f\"{k} {v:,}\" for k, v in ch.items()))\n            L.append(\"\")\n    L += [\"\", \"Метки — наблюдаемые события журнала (потеря связи, обесточивание, технические значения, газ ≥ 1 %), \"\n          \"не подтверждённые физические отказы. Правила: docs/EVENT_PANEL_V3_RU.md.\", \"\"]\n    return \"\\n\".join(L)\n\n\ndef run_event_panel_v3(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t_start = time.time()\n    out = Path(cfg[\"output_dir\"])\n    panel_dir = out / \"event_panel_v3\"\n    shutil.rmtree(panel_dir, ignore_errors=True)\n    panel_dir.mkdir(parents=True)\n    src = discover_sources(Path(cfg[\"input_dir\"]))\n    if not src[\"journals\"] or src[\"catalogue\"] is None:\n        return {\"status\": \"stopped\", \"stop_reason\": \"не найдены ext-journal-YYYY.* или \" + CATALOGUE_NAME\n                + \" во входах (Add Input → приватный dataset)\"}\n    smoke = cfg[\"mode\"] == \"SMOKE\"\n    years = cfg[\"smoke_years\"] if smoke else None\n    log(f\"входы: годы {list(src['journals'])}, справочник найден; режим {cfg['mode']}\")\n    year_files, stage = stage_journals(pl, src[\"journals\"], Path(cfg[\"stage_dir\"]), years, log)\n    if not year_files:\n        return {\"status\": \"stopped\", \"stop_reason\": \"после отбора годов не осталось журналов\"}\n    channel_filter = (pl.col(\"ид_канала_данных\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0) if smoke else None\n    _, audit = build_event_panel(pl, year_files, src[\"catalogue\"], Path(cfg[\"taxonomy_path\"]), config=cfg[\"panel\"],\n                                 log=log, n_buckets=cfg[\"n_buckets\"], out_dir=panel_dir, channel_filter=channel_filter)\n    _truncate_unknown(audit)\n    audit[\"sensitivity\"] = {}\n    key_cols = [\"d_channel_key\", \"d_cutoff_date\"] + TARGETS\n    main_labels = pl.read_parquet([panel_dir / p for p in audit[\"part_files\"]], columns=key_cols)\n    for name in cfg.get(\"sensitivity_variants\", []):\n        vdir = Path(cfg[\"stage_dir\"]) / f\"sensitivity_{name}\"\n        shutil.rmtree(vdir, ignore_errors=True)\n        vdir.mkdir(parents=True)\n        log(f\"sensitivity «{name}»: пересборка разметки\")\n        _, va = build_event_panel(pl, year_files, src[\"catalogue\"], Path(cfg[\"taxonomy_path\"]),\n                                  config={**cfg[\"panel\"], **SENSITIVITY_VARIANTS[name]}, log=lambda *a: None,\n                                  n_buckets=cfg[\"n_buckets\"], out_dir=vdir, channel_filter=channel_filter)\n        var_labels = pl.read_parquet([vdir / p for p in va[\"part_files\"]], columns=key_cols)\n        audit[\"sensitivity\"][name] = {\"config\": SENSITIVITY_VARIANTS[name],\n                                      \"label_changes\": label_changes(pl, main_labels, var_labels),\n                                      \"targets\": {t: {k: v for k, v in va[\"targets\"][t].items() if k != \"by_year\"}\n                                                  for t in TARGETS}}\n        shutil.rmtree(vdir, ignore_errors=True)\n    del main_labels\n    parts = []\n    for name in audit[\"part_files\"]:\n        p = panel_dir / name\n        parts.append({\"file\": name, \"bytes\": p.stat().st_size, \"sha256\": file_sha256(p),\n                      \"rows\": int(pl.scan_parquet(p).select(pl.len()).collect().item())})\n    manifest = {\n        \"schema_version\": SCHEMA_VERSION, \"created_at_utc\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\"),\n        \"mode\": cfg[\"mode\"], \"status\": \"completed_smoke_non_comparable\" if smoke else \"completed\",\n        \"config\": {k: v for k, v in cfg.items() if k not in (\"input_dir\", \"output_dir\", \"stage_dir\", \"taxonomy_path\")},\n        \"panel_config\": audit[\"config\"], \"taxonomy_version\": audit[\"taxonomy_version\"],\n        \"taxonomy_sha256\": audit[\"taxonomy_sha256\"], \"data_end\": audit[\"data_end\"],\n        \"catalogue\": {\"bytes\": src[\"catalogue\"].stat().st_size, \"sha256\": file_sha256(src[\"catalogue\"]),\n                      \"candidates_found\": src[\"catalogue_candidates\"]},\n        \"sources_ignored_as_duplicates\": src[\"ignored_duplicates\"], \"missing_years\": src[\"missing_years\"],\n        \"stage\": stage, \"parts\": parts,\n        \"rows\": audit[\"rows\"], \"channels\": audit[\"channels\"],\n        \"key_columns\": [\"d_channel_key\", \"d_cutoff_date\"], \"feature_columns\": audit[\"feature_columns\"],\n        \"target_columns\": audit[\"target_columns\"],\n        \"reason_columns\": [\"reason_\" + t[len(\"target_\"):] for t in audit[\"target_columns\"]],\n        \"purge_column\": \"d_label_decision_end\", \"polars_version\": pl.__version__,\n        \"runtime_s\": round(time.time() - t_start, 1),\n    }\n    (panel_dir / \"panel_manifest_v3.json\").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    (out / \"audit_event_panel_v3.json\").write_text(json.dumps(audit, ensure_ascii=False, indent=2, default=str), encoding=\"utf-8\")\n    (out / \"summary_event_panel_v3_ru.md\").write_text(summary_markdown(manifest, audit), encoding=\"utf-8\")\n    if not cfg[\"keep_stage\"]:\n        shutil.rmtree(cfg[\"stage_dir\"], ignore_errors=True)\n    return {\"status\": manifest[\"status\"], \"rows\": audit[\"rows\"], \"channels\": audit[\"channels\"],\n            \"runtime_s\": manifest[\"runtime_s\"], \"panel_dir\": str(panel_dir)}\n",
"gas_fingerprint_v4.py": "\"\"\"Notebook 19: aggregate diagnostics of gas crossings around suspected maintenance.\n\nThis module does not train or score a model.  It reuses the event-panel-v3\nclassification contract, detects strict upward crossings at second resolution,\ndescribes their observable shape, and evaluates whether there is enough support\noutside post-hoc suspected-maintenance windows to justify a separate stage 20.\n\nOnly aggregate results may leave the runtime.  Channel/object keys are used\ninternally and are never serialized.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport time\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nimport event_panel_v3 as ep\nimport event_panel_v3_kaggle as kag\n\nTARGET = \"target_t4_gas_cross\"\nEXPECTED_PANEL_MANIFEST_PREFIX = \"8cb6e0e2584f\"\nPERIODS = (\n    (\"le2022\", None, date(2023, 1, 1)),\n    (\"2023H1\", date(2023, 1, 1), date(2023, 7, 1)),\n    (\"2023H2\", date(2023, 7, 1), date(2024, 1, 1)),\n    (\"2024H1\", date(2024, 1, 1), date(2024, 7, 1)),\n    (\"2024H2\", date(2024, 7, 1), date(2025, 1, 1)),\n    (\"2025H1\", date(2025, 1, 1), date(2025, 7, 1)),\n    (\"2025H2\", date(2025, 7, 1), date(2026, 1, 1)),\n    (\"2026H1\", date(2026, 1, 1), date(2026, 7, 1)),\n)\nDEV_PERIODS = (\"2023H2\", \"2024H1\", \"2024H2\")\n# Audit 17 v2 (same frozen panel): positives in / outside the post-hoc window. Windows there were built from panel\n# gas_max, here from raw gas seconds; 17 also required d_label_decision_end <= period end. Small drift is expected.\nREFERENCE_17 = {\"2025H2\": {\"in_window\": 531, \"outside_window\": 33}, \"2026H1\": {\"in_window\": 226, \"outside_window\": 117}}\nSANITY_TOLERANCE = 0.10\n\n\nclass PanelMismatch(ValueError):\n    \"\"\"The supplied panel is not the frozen notebook-14 output expected by v3.\"\"\"\n\n\ndef make_config_19(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    if mode not in (\"FULL\", \"SMOKE\"):\n        raise ValueError(\"MODE must be FULL or SMOKE\")\n    cfg: dict[str, Any] = {\n        \"mode\": mode,\n        \"input_dir\": os.environ.get(\"LDT_KAGGLE_INPUT_DIR\", \"/kaggle/input\"),\n        \"output_dir\": os.environ.get(\"LDT_OUTPUT_DIR\", \"/kaggle/working\"),\n        \"stage_dir\": os.environ.get(\"LDT_STAGE_DIR\", \"/tmp/ldt_gas_fingerprint_19\"),\n        \"taxonomy_path\": None,\n        \"threshold\": 1.0,\n        \"rise_threshold\": 0.5,\n        \"peak_window_minutes\": 60,\n        \"duration_cap_minutes\": 1440,\n        \"min_silence_days\": 4,\n        \"before_days\": 3,\n        \"after_days\": 21,\n        \"expected_manifest_prefix\": EXPECTED_PANEL_MANIFEST_PREFIX,\n        \"verify_sha256\": True,\n        \"smoke_years\": [2023, 2024],\n        \"smoke_channel_share\": 20,\n        # Bound peak RAM: every bucket contains complete cross-year histories for its channels.\n        \"channel_buckets\": 64,\n        \"keep_stage\": False,\n    }\n    cfg.update(overrides or {})\n    return cfg\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        while block := f.read(1 << 22):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef discover_panel(input_dir: Path, verify: bool = True,\n                   expected_prefix: str | None = EXPECTED_PANEL_MANIFEST_PREFIX) -> tuple[list[Path], dict[str, Any], Path]:\n    manifests = sorted(Path(input_dir).rglob(\"panel_manifest_v3.json\"))\n    if not manifests:\n        raise FileNotFoundError(\"panel_manifest_v3.json not found: attach the frozen notebook-14 output\")\n    path = manifests[0]\n    digest = _sha256(path)\n    if expected_prefix and not digest.startswith(expected_prefix):\n        raise PanelMismatch(\n            f\"panel manifest {digest} is not the frozen v3 panel (expected prefix {expected_prefix}); do not bypass preflight\"\n        )\n    manifest = json.loads(path.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"schema_version\") != ep.SCHEMA_VERSION:\n        raise PanelMismatch(f\"panel schema {manifest.get('schema_version')} != {ep.SCHEMA_VERSION}\")\n    parts: list[Path] = []\n    for item in manifest.get(\"parts\", []):\n        part = path.parent / item[\"file\"]\n        if not part.is_file():\n            raise PanelMismatch(f\"missing panel part: {item['file']}\")\n        if verify and _sha256(part) != item.get(\"sha256\"):\n            raise PanelMismatch(f\"SHA-256 mismatch for panel part: {item['file']}\")\n        parts.append(part)\n    if not parts:\n        raise PanelMismatch(\"panel manifest contains no parts\")\n    return parts, manifest, path\n\n\ndef preflight_sources(cfg: dict[str, Any]) -> dict[str, Any]:\n    raw = kag.discover_sources(Path(cfg[\"input_dir\"]))\n    if not raw[\"journals\"] or raw[\"catalogue\"] is None:\n        raise FileNotFoundError(\"raw ext-journal-YYYY inputs and the sensor-channel catalogue are required\")\n    parts, manifest, path = discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"],\n                                            cfg.get(\"expected_manifest_prefix\"))\n    return {\n        \"journal_years\": list(raw[\"journals\"]),\n        \"catalogue_found\": True,\n        \"panel_manifest_sha256\": _sha256(path),\n        \"panel_rows\": int(manifest[\"rows\"]),\n        \"panel_parts\": len(parts),\n    }\n\n\ndef gas_seconds_from_raw(pl: Any, raw: Any, catalogue: Any, taxonomy: dict[str, Any]) -> Any:\n    \"\"\"Return one unordered-value-set row per gas channel/second.\"\"\"\n    gas_type = taxonomy[\"numeric\"][\"gas_sensor_type\"]\n    gas_ids = (catalogue.filter(pl.col(\"тип_датчика\") == gas_type)\n               .select(\"ид_канала_данных\").drop_nulls().unique())\n    if gas_ids.is_empty():\n        return pl.DataFrame(schema={\"d_channel_key\": pl.String, \"d_object_key\": pl.String,\n                                    \"t\": pl.Datetime(\"us\"), \"mn\": pl.Float64, \"mx\": pl.Float64})\n    subset = raw.join(gas_ids, on=\"ид_канала_данных\", how=\"semi\")\n    ev = ep.classify_events(pl, subset, taxonomy, catalogue).filter(\n        (pl.col(\"cls\") == \"numeric\") & pl.col(\"is_gas\") & pl.col(\"ид_объект\").is_not_null()\n    )\n    # Pseudonyms are computed once per unique id (as in event_panel_v3), not per row: tens of millions of rows.\n    ev = ev.with_columns(pl.col(\"ch\").cast(pl.String), pl.col(\"ид_объект\").cast(pl.String))\n    ch_map = ep.key_map(pl, ev[\"ch\"].unique().to_list(), \"d_channel_key\").rename({\"raw\": \"ch\"})\n    obj_map = ep.key_map(pl, ev[\"ид_объект\"].unique().to_list(), \"d_object_key\").rename({\"raw\": \"ид_объект\"})\n    ev = ev.join(ch_map, on=\"ch\", how=\"left\").join(obj_map, on=\"ид_объект\", how=\"left\")\n    return (ev.group_by(\"d_channel_key\", \"d_object_key\", \"t\").agg(\n        pl.col(\"num\").min().alias(\"mn\"), pl.col(\"num\").max().alias(\"mx\")\n    ))\n\n\ndef strict_crossing_features(pl: Any, seconds: Any, threshold: float = 1.0, rise_threshold: float = 0.5,\n                             peak_window_minutes: int = 60, duration_cap_minutes: int = 1440) -> Any:\n    \"\"\"Detect strict crossings and calculate shape features from second-level value sets.\n\n    A second containing values on both sides of the threshold is never strict.\n    The previous second must be unambiguously below the threshold.  Durations are\n    capped at 24h and explicitly marked as censored.\n    \"\"\"\n    schema = {\"d_channel_key\": pl.String, \"d_object_key\": pl.String, \"cross_t\": pl.Datetime(\"us\"),\n              \"day\": pl.Date, \"peak_60m\": pl.Float64, \"minutes_ge_1\": pl.Float64,\n              \"duration_censored\": pl.Boolean, \"rise_minutes\": pl.Float64, \"hour\": pl.Int8,\n              \"is_weekend\": pl.Boolean}\n    if seconds.is_empty():\n        return pl.DataFrame(schema=schema)\n    rows: list[dict[str, Any]] = []\n    peak_s = peak_window_minutes * 60\n    cap_s = duration_cap_minutes * 60\n    for group in seconds.sort(\"d_channel_key\", \"t\").partition_by(\"d_channel_key\", maintain_order=True):\n        times = group[\"t\"].to_numpy().astype(\"datetime64[us]\").astype(np.int64)\n        mn = group[\"mn\"].to_numpy()\n        mx = group[\"mx\"].to_numpy()\n        objects = group[\"d_object_key\"].to_list()\n        channels = group[\"d_channel_key\"].to_list()\n        strict = np.flatnonzero((mn[1:] >= threshold) & (mx[:-1] < threshold)) + 1\n        low_idx = np.flatnonzero(mx < rise_threshold)\n        for i in strict:\n            t0 = int(times[i])\n            peak_end = int(np.searchsorted(times, t0 + peak_s * 1_000_000, side=\"right\"))\n            peak = float(np.nanmax(mx[i:peak_end]))\n            duration_end = int(np.searchsorted(times, t0 + cap_s * 1_000_000, side=\"right\"))\n            drop_rel = np.flatnonzero(mn[i + 1:duration_end] < threshold)\n            if len(drop_rel):\n                j = i + 1 + int(drop_rel[0])\n                duration = (int(times[j]) - t0) / 60_000_000\n                censored = False\n            else:\n                duration = float(duration_cap_minutes)\n                censored = True\n            k = int(np.searchsorted(low_idx, i)) - 1        # last reading < rise_threshold before i (low_idx sorted)\n            rise = None if k < 0 else (t0 - int(times[int(low_idx[k])])) / 60_000_000\n            dt = group[\"t\"][int(i)]\n            rows.append({\n                \"d_channel_key\": channels[i], \"d_object_key\": objects[i], \"cross_t\": dt,\n                \"day\": dt.date(), \"peak_60m\": peak, \"minutes_ge_1\": float(duration),\n                \"duration_censored\": censored, \"rise_minutes\": None if rise is None else float(rise),\n                \"hour\": dt.hour, \"is_weekend\": dt.weekday() >= 5,\n            })\n    return pl.DataFrame(rows, schema=schema) if rows else pl.DataFrame(schema=schema)\n\n\ndef bounded_silence_runs_from_activity(pl: Any, observed: Any, min_silence_days: int = 4) -> Any:\n    \"\"\"Bounded gaps between observed object-days; no leading/trailing invented gaps.\"\"\"\n    schema = {\"d_object_key\": pl.String, \"start\": pl.Date, \"end\": pl.Date, \"length\": pl.Int32}\n    if observed.is_empty():\n        return pl.DataFrame(schema=schema)\n    observed = observed.unique().sort(\"d_object_key\", \"day\")\n    rows: list[dict[str, Any]] = []\n    for group in observed.partition_by(\"d_object_key\", maintain_order=True):\n        days = group[\"day\"].to_list()\n        obj = group[\"d_object_key\"][0]\n        for left, right in zip(days, days[1:]):\n            length = (right - left).days - 1\n            if length >= min_silence_days:\n                rows.append({\"d_object_key\": obj, \"start\": left + timedelta(days=1),\n                             \"end\": right - timedelta(days=1), \"length\": length})\n    return pl.DataFrame(rows, schema=schema) if rows else pl.DataFrame(schema=schema)\n\n\ndef bounded_silence_runs(pl: Any, seconds: Any, min_silence_days: int = 4) -> Any:\n    \"\"\"Bounded gaps between real object gas-reading days; no leading/trailing invented gaps.\"\"\"\n    if seconds.is_empty():\n        observed = pl.DataFrame(schema={\"d_object_key\": pl.String, \"day\": pl.Date})\n    else:\n        observed = seconds.select(\"d_object_key\", pl.col(\"t\").dt.date().alias(\"day\")).unique()\n    return bounded_silence_runs_from_activity(pl, observed, min_silence_days)\n\n\ndef process_seconds_buckets(pl: Any, paths: list[Path], cfg: dict[str, Any], log: Any = print) -> tuple[Any, Any]:\n    \"\"\"Process complete channel histories in bounded-RAM hash buckets.\"\"\"\n    n_buckets = int(cfg[\"channel_buckets\"])\n    if not 1 <= n_buckets <= 65_535:\n        raise ValueError(\"channel_buckets must be between 1 and 65535\")\n    crossing_frames = []\n    activity_frames = []\n    scans = [pl.scan_parquet(path) for path in paths]\n    for bucket in range(n_buckets):\n        seconds = (pl.concat(scans, how=\"vertical_relaxed\")\n                   .filter(pl.col(\"_channel_bucket\") == bucket)\n                   .collect()\n                   .sort(\"d_channel_key\", \"t\"))\n        if seconds.is_empty():\n            continue\n        crossing_frames.append(strict_crossing_features(\n            pl, seconds, cfg[\"threshold\"], cfg[\"rise_threshold\"],\n            cfg[\"peak_window_minutes\"], cfg[\"duration_cap_minutes\"],\n        ))\n        activity_frames.append(\n            seconds.select(\"d_object_key\", pl.col(\"t\").dt.date().alias(\"day\")).unique()\n        )\n        log(f\"channel bucket {bucket + 1}/{n_buckets}: {seconds.height:,} gas seconds\")\n        del seconds\n        gc.collect()\n    empty_seconds = pl.DataFrame(schema={\"d_channel_key\": pl.String, \"d_object_key\": pl.String,\n                                         \"t\": pl.Datetime(\"us\"), \"mn\": pl.Float64, \"mx\": pl.Float64})\n    crossings = (pl.concat(crossing_frames, how=\"vertical_relaxed\") if crossing_frames\n                 else strict_crossing_features(pl, empty_seconds))\n    activity = (pl.concat(activity_frames, how=\"vertical_relaxed\").unique()\n                if activity_frames else\n                pl.DataFrame(schema={\"d_object_key\": pl.String, \"day\": pl.Date}))\n    return crossings, activity\n\n\ndef add_maintenance_context(pl: Any, crossings: Any, runs: Any, before_days: int = 3,\n                            after_days: int = 21) -> Any:\n    \"\"\"Attach post-hoc strata and causal time-since-return at the crossing instant.\"\"\"\n    if crossings.is_empty():\n        return crossings.with_columns(pl.lit(False).alias(\"in_posthoc_window\"),\n                                      pl.lit(None, dtype=pl.Int32).alias(\"days_since_silence_end\"),\n                                      pl.lit(False).alias(\"silence_ongoing_at_event\"))\n    by_object = {g[\"d_object_key\"][0]: g.sort(\"start\") for g in runs.partition_by(\"d_object_key\")}\n    flags: list[bool] = []\n    ages: list[int | None] = []\n    for obj, day in crossings.select(\"d_object_key\", \"day\").iter_rows():\n        group = by_object.get(obj)\n        if group is None:\n            flags.append(False)\n            ages.append(None)\n            continue\n        in_window = False\n        latest_end: date | None = None\n        for start, end in group.select(\"start\", \"end\").iter_rows():\n            if start - timedelta(days=before_days) <= day <= end + timedelta(days=after_days):\n                in_window = True\n            # The crossing itself is a real reading, so a run ending before its day is known by the event time.\n            if end < day and (latest_end is None or end > latest_end):\n                latest_end = end\n        flags.append(in_window)\n        ages.append(None if latest_end is None else (day - latest_end).days)\n    return crossings.with_columns(\n        pl.Series(\"in_posthoc_window\", flags),\n        pl.Series(\"days_since_silence_end\", ages, dtype=pl.Int32),\n        # Always False by construction: the crossing itself is a real gas reading, so the object is not silent at\n        # the event instant. Kept as an explicit column for the stage-20 feature contract (003 A3).\n        pl.lit(False).alias(\"silence_ongoing_at_event\"),\n    )\n\n\ndef add_same_object_count(pl: Any, crossings: Any) -> Any:\n    if crossings.is_empty():\n        return crossings.with_columns(pl.lit(0, dtype=pl.Int32).alias(\"same_object_same_day\"))\n    counts = (crossings.group_by(\"d_object_key\", \"day\")\n              .agg(pl.col(\"d_channel_key\").n_unique().cast(pl.Int32).alias(\"same_object_same_day\")))\n    return crossings.join(counts, on=[\"d_object_key\", \"day\"], how=\"left\")\n\n\ndef period_name(day: date) -> str | None:\n    for name, start, end in PERIODS:\n        if (start is None or day >= start) and day < end:\n            return name\n    return None\n\n\ndef _hist(values: list[float | int | None], edges: list[float], labels: list[str]) -> dict[str, int]:\n    out = {label: 0 for label in labels}\n    for value in values:\n        if value is None or (isinstance(value, float) and math.isnan(value)):\n            out.setdefault(\"missing\", 0)\n            out[\"missing\"] += 1\n            continue\n        idx = int(np.searchsorted(np.asarray(edges), float(value), side=\"right\"))\n        out[labels[min(idx, len(labels) - 1)]] += 1\n    return out\n\n\ndef _quartiles(values: list[float | int | None]) -> dict[str, float | None]:\n    a = np.asarray([float(v) for v in values if v is not None and not math.isnan(float(v))], dtype=float)\n    if not len(a):\n        return {\"q1\": None, \"median\": None, \"q3\": None}\n    q = np.quantile(a, [0.25, 0.5, 0.75])\n    return {\"q1\": round(float(q[0]), 6), \"median\": round(float(q[1]), 6), \"q3\": round(float(q[2]), 6)}\n\n\ndef stratum_summary(frame: Any) -> dict[str, Any]:\n    peak = frame[\"peak_60m\"].to_list()\n    duration = frame[\"minutes_ge_1\"].to_list()\n    rise = frame[\"rise_minutes\"].to_list()\n    same = frame[\"same_object_same_day\"].to_list()\n    silence_age = frame[\"days_since_silence_end\"].to_list()\n    censored = frame[\"duration_censored\"].to_list()\n    duration_hist = {\"le5\": 0, \"gt5_le15\": 0, \"gt15_le60\": 0, \"gt60\": 0, \"censored\": 0}\n    for value, is_censored in zip(duration, censored):\n        if is_censored:\n            duration_hist[\"censored\"] += 1\n        elif value <= 5:\n            duration_hist[\"le5\"] += 1\n        elif value <= 15:\n            duration_hist[\"gt5_le15\"] += 1\n        elif value <= 60:\n            duration_hist[\"gt15_le60\"] += 1\n        else:\n            duration_hist[\"gt60\"] += 1\n    return {\n        \"events\": frame.height,\n        \"peak_60m_hist\": _hist(peak, [1.5, 1.9, 2.6, 3.8, 4.6],\n                                [\"1.0_1.5\", \"1.5_1.9\", \"1.9_2.6\", \"2.6_3.8\", \"3.8_4.6\", \"gt4.6\"]),\n        \"minutes_ge_1_hist\": duration_hist,\n        \"rise_minutes_hist\": _hist(rise, [1, 5, 15, 60, 1440],\n                                   [\"le1\", \"gt1_le5\", \"gt5_le15\", \"gt15_le60\", \"gt60_le1440\", \"gt1440\"]),\n        \"days_since_silence_end_hist\": _hist(\n            silence_age, [1, 7, 21, 60], [\"le1\", \"gt1_le7\", \"gt7_le21\", \"gt21_le60\", \"gt60\"]\n        ),\n        \"hour_hist\": {str(h): int(frame.filter(frame[\"hour\"] == h).height) for h in range(24)},\n        \"weekday\": int(frame.filter(~frame[\"is_weekend\"]).height),\n        \"weekend\": int(frame.filter(frame[\"is_weekend\"]).height),\n        \"same_object_same_day_hist\": _hist(same, [1.5, 2.5, 3.5], [\"1\", \"2\", \"3\", \"ge4\"]),\n        \"quartiles\": {\"peak_60m\": _quartiles(peak), \"minutes_ge_1\": _quartiles(duration),\n                      \"rise_minutes\": _quartiles(rise), \"days_since_silence_end\": _quartiles(silence_age),\n                      \"same_object_same_day\": _quartiles(same)},\n    }\n\n\ndef fit_fingerprint(events: Any) -> dict[str, Any]:\n    train = events.filter((events[\"day\"] < date(2025, 1, 1)) & events[\"in_posthoc_window\"]\n                          & (events[\"peak_60m\"] >= 1.5) & (events[\"peak_60m\"] <= 5.0))\n    if train.is_empty():\n        return {\"status\": \"insufficient_events\", \"n\": 0}\n    peaks = train[\"peak_60m\"].to_numpy()\n    bin_index = np.minimum(np.floor((peaks - 1.5) / 0.1).astype(int), 34)\n    counts = np.bincount(bin_index, minlength=35)\n    winner = int(np.flatnonzero(counts == counts.max())[0])\n    center = 1.5 + (winner + 0.5) * 0.1\n    q1, q3 = np.quantile(peaks, [0.25, 0.75])\n    durations = np.sort(train[\"minutes_ge_1\"].to_numpy())\n    rank = max(1, int(math.ceil(0.95 * len(durations)))) - 1\n    p95 = float(durations[rank])\n    censored_share = float(train[\"duration_censored\"].mean())\n    return {\n        \"status\": \"locked\", \"n\": train.height, \"training_cutoff\": \"2025-01-01\",\n        \"modal_bin_low\": round(1.5 + winner * 0.1, 6), \"modal_center\": round(center, 6),\n        \"plateau_q1\": round(float(q1), 6), \"plateau_q3\": round(float(q3), 6),\n        \"max_minutes_p95_nearest_rank\": round(p95, 6), \"censored_share\": round(censored_share, 6),\n        \"duration_is_lower_bound\": bool(p95 >= 1440 or censored_share >= 0.05),\n    }\n\n\ndef apply_fingerprint(pl: Any, events: Any, fingerprint: dict[str, Any]) -> Any:\n    if fingerprint.get(\"status\") != \"locked\":\n        return events.with_columns(pl.lit(False).alias(\"strict_fingerprint_match\"))\n    return events.with_columns(\n        (pl.col(\"peak_60m\").is_between(fingerprint[\"plateau_q1\"], fingerprint[\"plateau_q3\"], closed=\"both\")\n         & (pl.col(\"minutes_ge_1\") <= fingerprint[\"max_minutes_p95_nearest_rank\"]))\n        .alias(\"strict_fingerprint_match\")\n    )\n\n\ndef aggregate_crossings(pl: Any, events: Any) -> dict[str, Any]:\n    out: dict[str, Any] = {}\n    for name, _, _ in PERIODS:\n        p = events.filter(pl.col(\"period\") == name)\n        out[name] = {}\n        for window, flag in ((\"in_window\", True), (\"out_window\", False)):\n            s = p.filter(pl.col(\"in_posthoc_window\") == flag)\n            summary = stratum_summary(s)\n            if not flag:\n                matches = int(s[\"strict_fingerprint_match\"].sum()) if \"strict_fingerprint_match\" in s.columns else 0\n                summary[\"strict_fingerprint_matches\"] = matches\n                summary[\"strict_fingerprint_share\"] = matches / s.height if s.height else None\n            out[name][window] = summary\n    return out\n\n\ndef _posthoc_intervals(runs: Any, before: int, after: int) -> dict[str, list[tuple[date, date]]]:\n    out: dict[str, list[tuple[date, date]]] = {}\n    for obj, start, end in runs.select(\"d_object_key\", \"start\", \"end\").iter_rows():\n        out.setdefault(obj, []).append((start - timedelta(days=before), end + timedelta(days=after)))\n    return out\n\n\ndef panel_power(pl: Any, parts: list[Path], runs: Any, cfg: dict[str, Any]) -> dict[str, Any]:\n    lf = pl.scan_parquet(parts).filter(pl.col(TARGET) == 1)\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    positives = lf.select(\"d_object_key\", \"d_cutoff_date\").collect()\n    windows = _posthoc_intervals(runs, cfg[\"before_days\"], cfg[\"after_days\"])\n    rows: list[dict[str, Any]] = []\n    for obj, cutoff in positives.iter_rows():\n        outcome_day = cutoff + timedelta(days=2)\n        period = period_name(cutoff)\n        if period not in (*DEV_PERIODS, \"2025H2\", \"2026H1\"):\n            continue\n        inside = any(a <= outcome_day <= b for a, b in windows.get(obj, []))\n        rows.append({\"d_object_key\": obj, \"period\": period, \"in_posthoc_window\": inside})\n    result: dict[str, Any] = {}\n    for period in (*DEV_PERIODS, \"2025H2\", \"2026H1\"):\n        period_rows = [r for r in rows if r[\"period\"] == period]\n        group = [r[\"d_object_key\"] for r in period_rows if not r[\"in_posthoc_window\"]]\n        counts = sorted((group.count(obj) for obj in set(group)), reverse=True)\n        n = len(group)\n        result[period] = {\n            \"t_raw_positives\": len(period_rows),\n            \"positives_in_window_excluded_from_clean_target\": sum(r[\"in_posthoc_window\"] for r in period_rows),\n            \"positives_outside_window\": n,\n            \"objects_with_positive\": len(counts),\n            \"top1_object_share\": counts[0] / n if n else None,\n            \"top3_object_share\": sum(counts[:3]) / n if n else None,\n        }\n    dev_objects = [r[\"d_object_key\"] for r in rows if r[\"period\"] in DEV_PERIODS and not r[\"in_posthoc_window\"]]\n    dev_counts = sorted((dev_objects.count(obj) for obj in set(dev_objects)), reverse=True)\n    total = len(dev_objects)\n    top1 = dev_counts[0] / total if total else None\n    gate = {\n        \"dev_positives\": total,\n        \"dev_top1_object_share\": top1,\n        \"each_dev_at_least_10\": all(result[p][\"positives_outside_window\"] >= 10 for p in DEV_PERIODS),\n        \"sum_at_least_60\": total >= 60,\n        \"top1_at_most_50pct\": top1 is not None and top1 <= 0.5,\n    }\n    gate[\"proceed_to_stage20\"] = bool(gate[\"each_dev_at_least_10\"] and gate[\"sum_at_least_60\"]\n                                      and gate[\"top1_at_most_50pct\"])\n    gate[\"decision\"] = \"eligible_for_stage20_spec\" if gate[\"proceed_to_stage20\"] else \"insufficient_labels_keep_v3\"\n    return {\"periods\": result, \"gate_19_to_20\": gate, \"sanity_vs_17\": sanity_vs_17(result, cfg)}\n\n\ndef sanity_vs_17(periods: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"Compare window assignment with audit 17 v2; a large drift means the windows are built differently.\"\"\"\n    out: dict[str, Any] = {\"tolerance\": SANITY_TOLERANCE, \"comparable\": cfg[\"mode\"] == \"FULL\", \"periods\": {}}\n    for period, ref in REFERENCE_17.items():\n        got = {\"in_window\": periods[period][\"positives_in_window_excluded_from_clean_target\"],\n               \"outside_window\": periods[period][\"positives_outside_window\"]}\n        rel = {k: (abs(got[k] - v) / v if v else None) for k, v in ref.items()}\n        out[\"periods\"][period] = {\"stage19\": got, \"audit17\": ref, \"relative_diff\": rel,\n                                  \"within_tolerance\": all(r is not None and r <= SANITY_TOLERANCE for r in rel.values())}\n    out[\"all_within_tolerance\"] = all(p[\"within_tolerance\"] for p in out[\"periods\"].values())\n    return out\n\n\ndef _safe_result_text(result: dict[str, Any]) -> str:\n    text = json.dumps(result, ensure_ascii=False, indent=2, default=str)\n    forbidden = (\"d_channel_key\", \"d_object_key\", \"ид_канала\", \"ид_объект\")\n    if any(key in text for key in forbidden):\n        raise AssertionError(\"identifier field leaked into aggregate output\")\n    return text\n\n\ndef summary_markdown(result: dict[str, Any]) -> str:\n    gate = result[\"power\"][\"gate_19_to_20\"]\n    lines = [\"# Notebook 19 — диагностика газовых пересечений\", \"\",\n             \"Это диагностика наблюдаемых пересечений 1 % CH4, а не подтверждённых утечек или инцидентов.\",\n             \"Окна обозначают **подозреваемое обслуживание**; связь с поверкой не доказана.\", \"\",\n             f\"Статус: **{result['status']}**. Строгих пересечений: **{result['crossings_total']:,}**; \"\n             f\"ограниченных пауз всех газовых каналов объекта: **{result['silence_runs']:,}**.\", \"\",\n             \"## Gate 19 → 20\", \"\",\n             \"| период | T_raw = 1 | внутри окна (исключаются) | вне окна | объектов | top-1 | top-3 |\",\n             \"|---|---:|---:|---:|---:|---:|---:|\"]\n    for period, s in result[\"power\"][\"periods\"].items():\n        lines.append(f\"| {period} | {s['t_raw_positives']} | {s['positives_in_window_excluded_from_clean_target']} | \"\n                     f\"{s['positives_outside_window']} | {s['objects_with_positive']} | \"\n                     f\"{s['top1_object_share'] if s['top1_object_share'] is not None else '—'} | \"\n                     f\"{s['top3_object_share'] if s['top3_object_share'] is not None else '—'} |\")\n    sv = result[\"power\"].get(\"sanity_vs_17\")\n    if sv:\n        lines += [\"\", \"Сверка с аудитом 17 v2 (положительные в окне / вне окна): \" + \"; \".join(\n            f\"{p}: {v['stage19']['in_window']} / {v['stage19']['outside_window']} \"\n            f\"(17 v2: {v['audit17']['in_window']} / {v['audit17']['outside_window']})\"\n            for p, v in sv[\"periods\"].items())\n            + (\". Расхождение в пределах 10 %.\" if sv[\"all_within_tolerance\"] else\n               \". **ВНИМАНИЕ: расхождение > 10 % — окна построены иначе, чем в 17; решение по gate отложить до разбора.**\")\n            + (\"\" if sv[\"comparable\"] else \" (SMOKE — не сравнимо.)\")]\n    lines += [\"\", f\"Решение: **{gate['decision']}** (Σ dev = {gate['dev_positives']}, \"\n              f\"top-1 = {gate['dev_top1_object_share'] if gate['dev_top1_object_share'] is not None else '—'}).\", \"\",\n              \"Stage 20 не запускается автоматически: даже пройденный gate означает только достаточность для отдельной спецификации.\", \"\",\n              \"## Строгий отпечаток (зафиксирован только на ≤ 2024, внутри окон)\", \"\",\n              \"```json\", json.dumps(result[\"fingerprint_pre2025\"], ensure_ascii=False, indent=2), \"```\", \"\",\n              \"Полные агрегатные гистограммы и квартили находятся в `results_19_gas_fingerprint.json`. \"\n              \"T_raw и все исключаемые события сохраняются в агрегатных счётчиках; совпадение с отпечатком не исключает реальный инцидент.\", \"\"]\n    return \"\\n\".join(lines)\n\n\ndef run_stage19(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    started = time.time()\n    out = Path(cfg[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    stage = Path(cfg[\"stage_dir\"])\n    shutil.rmtree(stage, ignore_errors=True)\n    stage.mkdir(parents=True)\n    raw_sources = kag.discover_sources(Path(cfg[\"input_dir\"]))\n    if not raw_sources[\"journals\"] or raw_sources[\"catalogue\"] is None:\n        raise FileNotFoundError(\"raw journals or sensor catalogue not found\")\n    parts, manifest, manifest_path = discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"],\n                                                     cfg.get(\"expected_manifest_prefix\"))\n    years = cfg[\"smoke_years\"] if cfg[\"mode\"] == \"SMOKE\" else None\n    year_files, stage_audit = kag.stage_journals(pl, raw_sources[\"journals\"], stage, years, log)\n    taxonomy = ep.load_taxonomy(Path(cfg[\"taxonomy_path\"]))\n    catalogue = pl.read_csv(raw_sources[\"catalogue\"], infer_schema_length=0, encoding=\"utf8-lossy\")\n    gas_type = taxonomy[\"numeric\"][\"gas_sensor_type\"]\n    gas_ids = (catalogue.filter(pl.col(\"тип_датчика\") == gas_type)\n               .select(\"ид_канала_данных\").drop_nulls().unique().to_series().to_list())\n    if not gas_ids:\n        raise ValueError(\"the catalogue contains no gas channels\")\n    seconds_dir = stage / \"gas_seconds\"\n    seconds_dir.mkdir()\n    second_paths: list[Path] = []\n    gas_seconds_total = 0\n    for path in year_files:\n        # Scan the complete journal but collect only gas-channel rows; the full 313M-row archive never enters RAM.\n        raw = (pl.scan_parquet(path).select(ep.EVENT_COLUMNS)\n               .filter(pl.col(\"ид_канала_данных\").is_in(gas_ids)).collect())\n        seconds = gas_seconds_from_raw(pl, raw, catalogue, taxonomy)\n        if cfg[\"mode\"] == \"SMOKE\" and not seconds.is_empty():\n            seconds = seconds.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n        del raw\n        gc.collect()\n        # Sorting by the materialized bucket lets Parquet skip unrelated row groups on every bucket pass.\n        seconds = seconds.with_columns(\n            (pl.col(\"d_channel_key\").hash(seed=19) % cfg[\"channel_buckets\"])\n            .cast(pl.UInt16).alias(\"_channel_bucket\")\n        ).sort(\"_channel_bucket\", \"d_channel_key\", \"t\")\n        seconds_path = seconds_dir / f\"{path.stem}_gas_seconds.parquet\"\n        seconds.write_parquet(seconds_path, compression=\"zstd\", row_group_size=250_000)\n        second_paths.append(seconds_path)\n        gas_seconds_total += seconds.height\n        log(f\"{path.name}: gas seconds {seconds.height:,}\")\n        # The staged raw year is no longer needed; release both disk and memory before the next year.\n        path.unlink(missing_ok=True)\n        del seconds\n        gc.collect()\n    crossings, activity = process_seconds_buckets(pl, second_paths, cfg, log)\n    crossings = add_same_object_count(pl, crossings)\n    runs = bounded_silence_runs_from_activity(pl, activity, cfg[\"min_silence_days\"])\n    crossings = add_maintenance_context(pl, crossings, runs, cfg[\"before_days\"], cfg[\"after_days\"])\n    crossings = crossings.with_columns(\n        pl.col(\"day\").map_elements(period_name, return_dtype=pl.String).alias(\"period\")\n    ).filter(pl.col(\"period\").is_not_null())\n\n    fingerprint = fit_fingerprint(crossings)\n    partial = {\n        \"status\": \"fingerprint_locked\", \"analysis_version\": \"19-v1\",\n        \"panel\": {\"manifest_sha256\": _sha256(manifest_path), \"rows\": int(manifest[\"rows\"]),\n                  \"parts\": len(parts), \"schema_version\": manifest[\"schema_version\"]},\n        \"fingerprint_pre2025\": fingerprint,\n    }\n    result_path = out / \"results_19_gas_fingerprint.json\"\n    result_path.write_text(_safe_result_text(partial) + \"\\n\", encoding=\"utf-8\")\n    # Apply exactly the parameters already written above; late periods cannot refit them.\n    locked = json.loads(result_path.read_text(encoding=\"utf-8\"))[\"fingerprint_pre2025\"]\n    crossings = apply_fingerprint(pl, crossings, locked)\n    power = panel_power(pl, parts, runs, cfg)\n    result = {\n        \"status\": \"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\",\n        \"analysis_version\": \"19-v1\", \"mode\": cfg[\"mode\"],\n        \"runtime_s\": round(time.time() - started, 1),\n        \"panel\": partial[\"panel\"],\n        \"raw\": {\"journal_years\": sorted(int(y) for y in stage_audit.get(\"years\", {})),\n                \"gas_seconds\": gas_seconds_total},\n        \"crossings_total\": crossings.height, \"silence_runs\": runs.height,\n        \"fingerprint_pre2025\": locked,\n        \"crossing_strata\": aggregate_crossings(pl, crossings),\n        \"power\": power,\n        \"claims\": {\"event\": \"observable crossing of 1% CH4\", \"window\": \"suspected maintenance, not confirmed calibration\",\n                   \"incident_probability\": False, \"training_performed\": False},\n    }\n    result_path.write_text(_safe_result_text(result) + \"\\n\", encoding=\"utf-8\")\n    (out / \"summary_19_gas_fingerprint_ru.md\").write_text(summary_markdown(result), encoding=\"utf-8\")\n    if not cfg[\"keep_stage\"]:\n        shutil.rmtree(stage, ignore_errors=True)\n    return result\n",
"gas_triage_v1.py": "\"\"\"Research-only, causal-at-cutoff triage of observed CH4 crossings (spec 013+015).\n\nThe output of this module is an audit of hypotheses, not incident or maintenance\nground truth. No bundle, active model, or backend response is changed here.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport bisect\nimport gc\nimport json\nimport os\nimport random\nimport shutil\nimport time\nfrom collections import Counter, defaultdict\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nimport gas_fingerprint_v4 as gf\n\nLIVE_MINUTES = 15\nFAST_MINUTES = 5\nRECENT_MINUTES = 10\nPEAK_LIMIT = 2.6\nNEIGHBOR_HOURS = 6\nCONTEXT_DAYS = 90\nATTENTION = {\"needs_attention\", \"short_isolated\", \"pending\"}\nPERIODS = (\"2023H1\", \"2023H2\", \"2024H1\", \"2024H2\", \"2025H1\", \"2025H2\", \"2026H1\")\n\n\ndef make_config_21(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    cfg = gf.make_config_19(mode)\n    cfg.update({\"stage_dir\": os.environ.get(\"LDT_STAGE_DIR\", \"/tmp/ldt_gas_triage_21\"),\n                \"keep_stage\": False, \"analysis_version\": \"21-v1\"})\n    cfg.update(overrides or {})\n    return cfg\n\n\ndef preflight(cfg: dict[str, Any]) -> dict[str, Any]:\n    source = gf.preflight_sources(cfg)\n    stage_parent = Path(cfg[\"stage_dir\"]).resolve().parent\n    stage_parent.mkdir(parents=True, exist_ok=True)\n    disk = shutil.disk_usage(stage_parent)\n    try:\n        import psutil\n        ram_gib = round(psutil.virtual_memory().total / (1024 ** 3), 1)\n    except ImportError:\n        ram_gib = None\n    return {**source, \"mode\": cfg[\"mode\"], \"channel_buckets\": cfg[\"channel_buckets\"],\n            \"free_stage_gib\": round(disk.free / (1024 ** 3), 1), \"ram_gib\": ram_gib,\n            \"staging_note\": \"FULL rereads raw yearly journals and writes temporary gas-second Parquet\"}\n\n\ndef _event_shapes_for_bucket(pl: Any, seconds: Any) -> list[dict[str, Any]]:\n    \"\"\"Use Stage 19's strict detector; derive live shape only through t0+15m.\"\"\"\n    crossings = gf.strict_crossing_features(pl, seconds)\n    if crossings.is_empty():\n        return []\n    events_by_channel: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for event in crossings.iter_rows(named=True):\n        events_by_channel[event[\"d_channel_key\"]].append(event)\n    rows: list[dict[str, Any]] = []\n    for group in seconds.partition_by(\"d_channel_key\"):\n        channel_events = events_by_channel.get(group[\"d_channel_key\"][0], [])\n        if not channel_events:\n            continue\n        times = group[\"t\"].to_numpy().astype(\"datetime64[us]\").astype(np.int64)\n        mx = group[\"mx\"].to_numpy()\n        for event in channel_events:\n            t0 = event[\"cross_t\"]\n            t0_us = np.datetime64(t0, \"us\").astype(np.int64)\n            c_us = t0_us + LIVE_MINUTES * 60_000_000\n            lo = int(np.searchsorted(times, t0_us, side=\"left\"))\n            hi = int(np.searchsorted(times, c_us, side=\"right\"))\n            fast_hi = int(np.searchsorted(times, t0_us + FAST_MINUTES * 60_000_000, side=\"right\"))\n            drop_candidates = np.flatnonzero(mx[lo + 1:fast_hi] < 1.0)\n            drop_idx = lo + 1 + int(drop_candidates[0]) if len(drop_candidates) else None\n            repeat = bool(drop_idx is not None and np.any(mx[drop_idx + 1:hi] >= 1.0))\n            ended_fast = drop_idx is not None and not repeat\n            peak = float(np.nanmax(mx[lo:hi]))\n            valid = np.flatnonzero(np.isfinite(mx[lo:hi]))\n            last_idx = lo + int(valid[-1])\n            age_minutes = (c_us - int(times[last_idx])) / 60_000_000\n            state = (\"unknown\" if drop_idx is None or repeat or not np.isfinite(peak) else\n                     \"observed_below\" if age_minutes <= RECENT_MINUTES else \"inferred_below\")\n            rows.append({\"d_channel_key\": event[\"d_channel_key\"], \"d_object_key\": event[\"d_object_key\"],\n                         \"cross_t\": t0, \"day\": event[\"day\"], \"decision_t\": t0 + timedelta(minutes=LIVE_MINUTES),\n                         \"ended_fast\": ended_fast, \"repeat_by_c\": repeat, \"peak_by_c\": peak,\n                         \"last_reading_age_min\": float(age_minutes), \"state_at_c\": state,\n                         \"drop_t\": None if drop_idx is None else group[\"t\"][drop_idx]})\n    return rows\n\n\ndef _unique_other_count(events: list[dict[str, Any]], left: datetime, right: datetime,\n                        channel: str) -> int:\n    return len({e[\"d_channel_key\"] for e in events\n                if left <= e[\"cross_t\"] <= right and e[\"d_channel_key\"] != channel})\n\n\ndef _context(event: dict[str, Any], runs_by_object: dict[str, list[dict[str, Any]]],\n             first_day: dict[str, date]) -> tuple[str, bool]:\n    day = event[\"day\"]\n    obj = event[\"d_object_key\"]\n    first = first_day.get(obj)\n    if first is None or (day - first).days < CONTEXT_DAYS:\n        return \"unknown\", False\n    runs = runs_by_object.get(obj, [])\n    recent = any(1 <= (day - r[\"end\"]).days <= 21 for r in runs)\n    in_posthoc = any(r[\"start\"] - timedelta(days=3) <= day <= r[\"end\"] + timedelta(days=21)\n                     for r in runs)\n    return (\"true\" if recent else \"false\"), in_posthoc\n\n\ndef label_events(events: list[dict[str, Any]], activity: Any, runs: Any,\n                 archive_end: datetime) -> list[dict[str, Any]]:\n    \"\"\"Label compact crossing rows; all neighbor counts use the full object timeline.\"\"\"\n    by_object: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for event in events:\n        by_object[event[\"d_object_key\"]].append(event)\n    for group in by_object.values():\n        group.sort(key=lambda e: e[\"cross_t\"])\n    runs_by_object: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for r in runs.iter_rows(named=True):\n        runs_by_object[r[\"d_object_key\"]].append(r)\n    first_day: dict[str, date] = {}\n    for obj, day in activity.select(\"d_object_key\", \"day\").iter_rows():\n        first_day[obj] = min(day, first_day.get(obj, day))\n\n    for event in events:\n        t0, c, day = event[\"cross_t\"], event[\"decision_t\"], event[\"day\"]\n        group = by_object[event[\"d_object_key\"]]\n        event[\"other_ch_by_live_cutoff\"] = _unique_other_count(\n            group, t0 - timedelta(hours=NEIGHBOR_HOURS), c, event[\"d_channel_key\"])\n        event[\"other_ch_day\"] = _unique_other_count(\n            group, datetime.combine(day, datetime.min.time()),\n            datetime.combine(day + timedelta(days=1), datetime.min.time()) - timedelta(microseconds=1),\n            event[\"d_channel_key\"])\n        event[\"after_silence\"], event[\"in_posthoc_window\"] = _context(event, runs_by_object, first_day)\n        event[\"period\"] = gf.period_name(day)\n        event[\"is_weekend\"] = t0.weekday() >= 5\n        event[\"outside_07_19\"] = not 7 <= t0.hour < 19\n        bad_shape = (not event[\"ended_fast\"] or not np.isfinite(event[\"peak_by_c\"]) or\n                     event[\"peak_by_c\"] > PEAK_LIMIT or event[\"state_at_c\"] == \"unknown\")\n        event[\"shape_ok\"] = not bad_shape\n        if bad_shape:\n            label = \"needs_attention\"\n        elif event[\"after_silence\"] == \"true\":\n            label = \"likely_after_maintenance\"\n        elif event[\"other_ch_by_live_cutoff\"] >= 1:\n            label = \"likely_bump_test\"\n        else:\n            label = \"short_isolated\"\n        event[\"triage_live\"] = label if c <= archive_end else None\n        event[\"status_at_archive_end\"] = \"censored_right\" if c > archive_end else \"decided\"\n        event[\"status_at_day_end\"] = \"pending_at_day_end\" if c.date() > day else \"decided\"\n        day_end = datetime.combine(day + timedelta(days=1), datetime.min.time()) - timedelta(microseconds=1)\n        event[\"triage_retro\"] = (\"censored_right\" if day_end > archive_end else\n                                 \"censored_by_day\" if c > day_end else\n                                 \"needs_attention\" if bad_shape else\n                                 \"likely_after_maintenance\" if event[\"after_silence\"] == \"true\" else\n                                 \"likely_bump_test\" if event[\"other_ch_day\"] >= 2 else \"short_isolated\")\n        event[\"triage_live_strict\"] = (None if c > archive_end else\n                                       \"needs_attention\" if label.startswith(\"likely_\") and\n                                       event[\"state_at_c\"] != \"observed_below\" else label)\n        event[\"series_only\"] = bool(event[\"shape_ok\"] and event[\"other_ch_by_live_cutoff\"] >= 1)\n        event[\"series_only_strict\"] = bool(event[\"series_only\"] and event[\"state_at_c\"] == \"observed_below\")\n        event[\"reason_codes\"] = {\"ended_fast\": event[\"ended_fast\"], \"repeat_by_c\": event[\"repeat_by_c\"],\n                                 \"peak_by_c\": event[\"peak_by_c\"],\n                                 \"other_ch_by_live_cutoff\": event[\"other_ch_by_live_cutoff\"],\n                                 \"after_silence\": event[\"after_silence\"],\n                                 \"last_reading_age_min\": event[\"last_reading_age_min\"]}\n    return events\n\n\ndef _rate(rows: list[dict[str, Any]], field: str) -> float | None:\n    return round(sum(bool(r[field]) for r in rows) / len(rows), 6) if rows else None\n\n\ndef _label_summary(events: list[dict[str, Any]], label_field: str) -> dict[str, Any]:\n    by_label: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for e in events:\n        by_label[e[label_field] or \"pending\"].append(e)\n    result = {}\n    for label, rows in sorted(by_label.items()):\n        object_counts = Counter(e[\"d_object_key\"] for e in rows)\n        ranked = sorted(object_counts.values(), reverse=True)\n        result[label] = {\"n\": len(rows), \"weekend_share\": _rate(rows, \"is_weekend\"),\n                         \"outside_07_19_share\": _rate(rows, \"outside_07_19\"),\n                         \"top1_object_share\": round(ranked[0] / len(rows), 6),\n                         \"top3_object_share\": round(sum(ranked[:3]) / len(rows), 6)}\n    return result\n\n\ndef _bootstrap_calendar(events: list[dict[str, Any]], label_field: str,\n                        repeats: int = 2000) -> dict[str, Any]:\n    relevant = [e for e in events if e[label_field] == \"likely_bump_test\" or e[label_field] in ATTENTION]\n    blocks: dict[tuple[str, int, int], list[dict[str, Any]]] = defaultdict(list)\n    for e in relevant:\n        iso = e[\"day\"].isocalendar()\n        blocks[(e[\"d_object_key\"], iso.year, iso.week)].append(e)\n    rng = random.Random(21)\n    groups = list(blocks.values())\n    diffs: list[float] = []\n    for _ in range(repeats):\n        chosen = [row for _ in groups for row in groups[rng.randrange(len(groups))]] if groups else []\n        bump = [e for e in chosen if e[label_field] == \"likely_bump_test\"]\n        queue = [e for e in chosen if e[label_field] in ATTENTION]\n        if bump and queue:\n            diffs.append(sum(e[\"is_weekend\"] for e in queue) / len(queue) -\n                         sum(e[\"is_weekend\"] for e in bump) / len(bump))\n    return {\"repeats\": repeats, \"valid_repeats\": len(diffs),\n            \"weekend_queue_minus_bump_ci95\": ([round(float(np.quantile(diffs, .025)), 6),\n                                                 round(float(np.quantile(diffs, .975)), 6)] if diffs else None)}\n\n\ndef _n1_control(events: list[dict[str, Any]], archive_start: datetime,\n                archive_end: datetime, strict: bool = False) -> dict[str, Any]:\n    cohort = [e for e in events if e[\"period\"] in PERIODS[:-1]]\n    if not cohort:\n        return {\"n\": 0, \"passed\": False, \"reason\": \"empty_cohort\"}\n    # The observed and shifted fractions use exactly the same event cohort.\n    by_obj_observed: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for e in cohort:\n        by_obj_observed[e[\"d_object_key\"]].append(e)\n    true_share = sum((e[\"shape_ok\"] and (not strict or e[\"state_at_c\"] == \"observed_below\") and\n                      _unique_other_count(by_obj_observed[e[\"d_object_key\"]],\n                                          e[\"cross_t\"] - timedelta(hours=NEIGHBOR_HOURS),\n                                          e[\"decision_t\"], e[\"d_channel_key\"]) >= 1)\n                     for e in cohort) / len(cohort)\n    rng = random.Random(21)\n    channels = sorted({e[\"d_channel_key\"] for e in cohort})\n    fractions: list[float] = []\n    unshifted: list[int] = []\n    right = archive_end - timedelta(days=1)\n    for _ in range(20):\n        offsets = {ch: rng.choice((-2, -1, 1, 2)) * timedelta(weeks=1) for ch in channels}\n        by_channel = defaultdict(list)\n        for e in cohort:\n            by_channel[e[\"d_channel_key\"]].append(e[\"cross_t\"])\n        for ch, original_times in by_channel.items():\n            offset = offsets[ch]\n            if any(not archive_start <= t + offset <= right for t in original_times):\n                offsets[ch] = -offset\n        shifted: list[dict[str, Any]] = []\n        stationary = 0\n        for e in cohort:\n            t = e[\"cross_t\"] + offsets[e[\"d_channel_key\"]]\n            if not archive_start <= t <= right:\n                t = e[\"cross_t\"]\n                stationary += 1\n            shifted.append({\"d_object_key\": e[\"d_object_key\"], \"d_channel_key\": e[\"d_channel_key\"],\n                            \"cross_t\": t, \"shape_ok\": e[\"shape_ok\"], \"state_at_c\": e[\"state_at_c\"]})\n        by_obj: dict[str, list[dict[str, Any]]] = defaultdict(list)\n        for e in shifted:\n            by_obj[e[\"d_object_key\"]].append(e)\n        n_series = sum(e[\"shape_ok\"] and (not strict or e[\"state_at_c\"] == \"observed_below\") and\n            _unique_other_count(\n            by_obj[e[\"d_object_key\"]], e[\"cross_t\"] - timedelta(hours=NEIGHBOR_HOURS),\n            e[\"cross_t\"] + timedelta(minutes=LIVE_MINUTES), e[\"d_channel_key\"]) >= 1\n            for e in shifted)\n        fractions.append(n_series / len(cohort))\n        unshifted.append(stationary)\n    return {\"n\": len(cohort), \"seed\": 21, \"permutations\": 20, \"strict_observed_only\": strict,\n            \"observed_share\": round(true_share, 6),\n            \"permuted_median_share\": round(float(np.median(fractions)), 6),\n            \"permuted_max_share\": round(max(fractions), 6),\n            \"n1_unshifted_max\": max(unshifted),\n            \"passed\": true_share > max(fractions)}\n\n\ndef _v1_calendar(cohort: list[dict[str, Any]], label_field: str) -> dict[str, Any]:\n    bump = [e for e in cohort if e[label_field] == \"likely_bump_test\"]\n    attention = [e for e in cohort if e[label_field] in ATTENTION]\n    bump_weekend, queue_weekend = _rate(bump, \"is_weekend\"), _rate(attention, \"is_weekend\")\n    passed = bool(bump_weekend is not None and queue_weekend is not None and\n                  bump_weekend <= .05 and bump_weekend * 2 <= queue_weekend)\n    objects = {e[\"d_object_key\"] for e in cohort}\n    loo_passes = []\n    for obj in objects:\n        rows = [e for e in cohort if e[\"d_object_key\"] != obj]\n        b = [e for e in rows if e[label_field] == \"likely_bump_test\"]\n        q = [e for e in rows if e[label_field] in ATTENTION]\n        wb, wq = _rate(b, \"is_weekend\"), _rate(q, \"is_weekend\")\n        loo_passes.append(bool(wb is not None and wq is not None and wb <= .05 and 2 * wb <= wq))\n    return {\"cohort\": \"2023-2025\", \"bump_n\": len(bump), \"attention_n\": len(attention),\n            \"bump_weekend_share\": bump_weekend, \"attention_weekend_share\": queue_weekend,\n            \"bump_outside_07_19_share\": _rate(bump, \"outside_07_19\"),\n            \"attention_outside_07_19_share\": _rate(attention, \"outside_07_19\"),\n            \"bootstrap\": _bootstrap_calendar(cohort, label_field),\n            \"leave_one_object_out\": {\"objects\": len(objects), \"passes\": sum(loo_passes)},\n            \"passed\": passed}\n\n\ndef _daily_burden(events: list[dict[str, Any]], period: str,\n                  archive_end: datetime) -> dict[str, Any]:\n    period_def = next((start, end) for name, start, end in gf.PERIODS if name == period)\n    start, end = period_def\n    if start is None:\n        # The archive's early-years start is an observed bound, not an invented 2019-01-01.\n        start = min((e[\"day\"] for e in events), default=end)\n    end = min(end, archive_end.date() + timedelta(days=1))\n    days = max(1, (end - start).days)\n    total = Counter(e[\"day\"] for e in events)\n    queue = Counter(e[\"day\"] for e in events if\n                    e[\"status_at_day_end\"] == \"pending_at_day_end\" or\n                    e[\"status_at_archive_end\"] == \"censored_right\" or\n                    e[\"triage_live\"] in ATTENTION)\n    arr_all = [total[start + timedelta(days=i)] for i in range(days)]\n    arr_queue = [queue[start + timedelta(days=i)] for i in range(days)]\n    return {\"calendar_days\": days, \"all_mean_per_day\": round(float(np.mean(arr_all)), 3),\n            \"all_p95_per_day\": round(float(np.quantile(arr_all, .95)), 3),\n            \"attention_mean_per_day\": round(float(np.mean(arr_queue)), 3),\n            \"attention_p95_per_day\": round(float(np.quantile(arr_queue, .95)), 3)}\n\n\ndef aggregate_21(events: list[dict[str, Any]], runs_count: int, seconds_count: int,\n                 archive_start: datetime, archive_end: datetime, cfg: dict[str, Any],\n                 panel: dict[str, Any]) -> dict[str, Any]:\n    by_period = {name: [e for e in events if e[\"period\"] == name]\n                 for name, _, _ in gf.PERIODS}\n    cohort = [e for e in events if e[\"period\"] in PERIODS[:-1]]\n    v1 = _v1_calendar(cohort, \"triage_live\")\n    v1_strict = _v1_calendar(cohort, \"triage_live_strict\")\n    n1 = _n1_control(events, archive_start, archive_end)\n    n1_strict = _n1_control(events, archive_start, archive_end, strict=True)\n    bins = {\"le1\": 0, \"gt1_le5\": 0, \"gt5_le10\": 0,\n            \"gt10_le30\": 0, \"gt30_le120\": 0, \"gt120\": 0}\n    likely = [e for e in events if (e[\"triage_live\"] or \"\").startswith(\"likely_\")]\n    for e in likely:\n        age = e[\"last_reading_age_min\"]\n        key = (\"le1\" if age <= 1 else \"gt1_le5\" if age <= 5 else \"gt5_le10\" if age <= 10 else\n               \"gt10_le30\" if age <= 30 else \"gt30_le120\" if age <= 120 else \"gt120\")\n        bins[key] += 1\n    periods = {}\n    for name, rows in by_period.items():\n        if not rows:\n            continue\n        in_window = [e for e in rows if e[\"in_posthoc_window\"]]\n        periods[name] = {\"n\": len(rows), \"labels_live\": _label_summary(rows, \"triage_live\"),\n                         \"labels_live_strict\": _label_summary(rows, \"triage_live_strict\"),\n                         \"labels_retro\": _label_summary(rows, \"triage_retro\"),\n                         \"pending_at_day_end\": sum(e[\"status_at_day_end\"] == \"pending_at_day_end\" for e in rows),\n                         \"censored_right\": sum(e[\"status_at_archive_end\"] == \"censored_right\" for e in rows),\n                         \"unresolved_or_incomplete\": sum(not e[\"ended_fast\"] for e in rows),\n                         \"context_unknown\": sum(e[\"after_silence\"] == \"unknown\" for e in rows),\n                         \"observed_below\": sum(e[\"state_at_c\"] == \"observed_below\" for e in rows),\n                         \"inferred_below\": sum(e[\"state_at_c\"] == \"inferred_below\" for e in rows),\n                         \"retro_disagreements\": sum(e[\"triage_retro\"] not in (\"censored_by_day\", \"censored_right\") and\n                                                     e[\"triage_retro\"] != e[\"triage_live\"] for e in rows),\n                         \"v2_in_posthoc_window\": {\"n\": len(in_window),\n                             \"hypothesis_share\": round(sum((e[\"triage_live\"] or \"\").startswith(\"likely_\")\n                                                        for e in in_window) / len(in_window), 6) if in_window else None,\n                             \"series_only_share\": _rate(in_window, \"series_only\")},\n                         \"v6_daily_burden\": _daily_burden(rows, name, archive_end)}\n    prior = [e for e in cohort if e[\"triage_live\"] is not None]\n    late = [e for e in by_period[\"2026H1\"] if e[\"triage_live\"] is not None]\n    def attention_share(rows: list[dict[str, Any]]) -> float | None:\n        return round(sum(e[\"triage_live\"] in ATTENTION for e in rows) / len(rows), 6) if rows else None\n    return {\"status\": \"completed\" if cfg[\"mode\"] == \"FULL\" else \"completed_smoke_non_comparable\",\n            \"analysis_version\": cfg[\"analysis_version\"], \"mode\": cfg[\"mode\"],\n            \"panel\": panel, \"gas_seconds\": seconds_count,\n            \"crossings_total\": len(events), \"silence_runs\": runs_count,\n            \"v1_calendar\": v1, \"v1_calendar_strict\": v1_strict,\n            \"n1_synchrony_control\": n1, \"n1_synchrony_control_strict\": n1_strict,\n            \"likely_last_reading_age_bins\": bins,\n            \"v3_2026h1_observation\": {\"prior_2023_2025_attention_share\": attention_share(prior),\n                                      \"2026h1_attention_share\": attention_share(late),\n                                      \"exploratory_only\": True},\n            \"periods\": periods,\n            \"backend_gate\": \"research_only\" if v1[\"passed\"] and n1[\"passed\"] else \"stop_v1_or_n1\",\n            \"claims\": {\"incident_labels\": False, \"verified_calibration\": False,\n                       \"active_bundle_changed\": False, \"backend_deployed\": False}}\n\n\ndef summary_markdown(result: dict[str, Any]) -> str:\n    lines = [\"# Ноутбук 21 — исследовательский триаж газовых тревог\", \"\",\n             \"Метки совместимы с гипотезой проверки датчиков, но не подтверждают поверку или инцидент.\",\n             \"Тревоги не скрываются; решение о backend принимается отдельно.\", \"\",\n             f\"Статус: **{result['status']}**; строгих пересечений: **{result['crossings_total']:,}**; \"\n             f\"ограниченных пауз: **{result['silence_runs']:,}**.\", \"\",\n             \"## Предзаданные проверки\", \"\",\n             f\"- V1 календарь: **{'прошёл' if result['v1_calendar']['passed'] else 'не прошёл'}**.\",\n             f\"- N1 синхронизация: **{'прошёл' if result['n1_synchrony_control']['passed'] else 'не прошёл'}**.\",\n             f\"- Stop-rule: **{result['backend_gate']}**. Прохождение не доказывает истинную причину тревог.\", \"\",\n             \"## Метки по периодам\", \"\",\n             \"| Период | Событий | Needs attention | Short isolated | After maintenance | Bump test | Pending D |\",\n             \"|---|---:|---:|---:|---:|---:|---:|\"]\n    for name, p in result[\"periods\"].items():\n        labels = p[\"labels_live\"]\n        count = lambda label: labels.get(label, {}).get(\"n\", 0)\n        lines.append(f\"| {name} | {p['n']} | {count('needs_attention')} | {count('short_isolated')} | \"\n                     f\"{count('likely_after_maintenance')} | {count('likely_bump_test')} | {p['pending_at_day_end']} |\")\n    lines.extend([\"\", \"Полные агрегаты V1–V6, строгий вариант, цензура и нагрузка — в JSON. \"\n                  \"Идентификаторы объектов и каналов не публикуются.\"])\n    return \"\\n\".join(lines) + \"\\n\"\n\n\ndef run_stage21(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    started = time.time()\n    out = Path(cfg[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    parts, manifest, manifest_path = gf.discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"],\n                                                       cfg.get(\"expected_manifest_prefix\"))\n    del parts\n    paths, audit, _, seconds_count = _stage_seconds(pl, cfg, log)\n    events, activity, archive_start, archive_end = _process_buckets(pl, paths, cfg, log)\n    runs = gf.bounded_silence_runs_from_activity(pl, activity, cfg[\"min_silence_days\"])\n    events = label_events(events, activity, runs, archive_end)\n    events = [e for e in events if e[\"period\"] is not None]\n    if cfg[\"mode\"] == \"FULL\" and (len(events), runs.height) != (7_222, 93):\n        raise ValueError(f\"Stage 19 identity mismatch: crossings={len(events)}, runs={runs.height}; expected 7222/93\")\n    result = aggregate_21(events, runs.height, seconds_count, archive_start, archive_end, cfg,\n                          {\"manifest_sha256\": gf._sha256(manifest_path), \"rows\": int(manifest[\"rows\"]),\n                           \"parts\": len(manifest[\"parts\"]), \"schema_version\": manifest[\"schema_version\"],\n                           \"journal_years\": sorted(int(y) for y in audit.get(\"years\", {}))})\n    result[\"runtime_s\"] = round(time.time() - started, 1)\n    (out / \"results_21_gas_triage.json\").write_text(gf._safe_result_text(result) + \"\\n\", encoding=\"utf-8\")\n    (out / \"summary_21_gas_triage_ru.md\").write_text(summary_markdown(result), encoding=\"utf-8\")\n    if not cfg[\"keep_stage\"]:\n        shutil.rmtree(Path(cfg[\"stage_dir\"]), ignore_errors=True)\n    return result\n\n\ndef _stage_seconds(pl: Any, cfg: dict[str, Any], log: Any) -> tuple[list[Path], dict[str, Any], Any, int]:\n    stage = Path(cfg[\"stage_dir\"])\n    shutil.rmtree(stage, ignore_errors=True)\n    stage.mkdir(parents=True)\n    raw_sources = gf.kag.discover_sources(Path(cfg[\"input_dir\"]))\n    if not raw_sources[\"journals\"] or raw_sources[\"catalogue\"] is None:\n        raise FileNotFoundError(\"raw journals or catalogue not found\")\n    years = cfg[\"smoke_years\"] if cfg[\"mode\"] == \"SMOKE\" else None\n    year_files, audit = gf.kag.stage_journals(pl, raw_sources[\"journals\"], stage, years, log)\n    taxonomy = gf.ep.load_taxonomy(Path(cfg[\"taxonomy_path\"]))\n    catalogue = pl.read_csv(raw_sources[\"catalogue\"], infer_schema_length=0, encoding=\"utf8-lossy\")\n    gas_type = taxonomy[\"numeric\"][\"gas_sensor_type\"]\n    gas_ids = (catalogue.filter(pl.col(\"тип_датчика\") == gas_type)\n               .select(\"ид_канала_данных\").drop_nulls().unique().to_series().to_list())\n    if not gas_ids:\n        raise ValueError(\"the catalogue contains no gas channels\")\n    seconds_dir = stage / \"gas_seconds\"\n    seconds_dir.mkdir()\n    paths: list[Path] = []\n    total = 0\n    for path in year_files:\n        raw = (pl.scan_parquet(path).select(gf.ep.EVENT_COLUMNS)\n               .filter(pl.col(\"ид_канала_данных\").is_in(gas_ids)).collect())\n        sec = gf.gas_seconds_from_raw(pl, raw, catalogue, taxonomy)\n        if cfg[\"mode\"] == \"SMOKE\" and not sec.is_empty():\n            sec = sec.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n        del raw\n        gc.collect()\n        sec = sec.with_columns((pl.col(\"d_channel_key\").hash(seed=19) % cfg[\"channel_buckets\"])\n                               .cast(pl.UInt16).alias(\"_channel_bucket\"))\n        sec = sec.sort(\"_channel_bucket\", \"d_channel_key\", \"t\")\n        out = seconds_dir / f\"{path.stem}_gas_seconds.parquet\"\n        sec.write_parquet(out, compression=\"zstd\", row_group_size=250_000)\n        paths.append(out)\n        total += sec.height\n        log(f\"{path.name}: gas seconds {sec.height:,}\")\n        path.unlink(missing_ok=True)\n        del sec\n        gc.collect()\n    return paths, audit, catalogue, total\n\n\ndef _process_buckets(pl: Any, paths: list[Path], cfg: dict[str, Any], log: Any\n                     ) -> tuple[list[dict[str, Any]], Any, datetime, datetime]:\n    scans = [pl.scan_parquet(path) for path in paths]\n    events: list[dict[str, Any]] = []\n    activity_frames = []\n    archive_end: datetime | None = None\n    archive_start: datetime | None = None\n    for bucket in range(int(cfg[\"channel_buckets\"])):\n        sec = (pl.concat(scans, how=\"vertical_relaxed\")\n               .filter(pl.col(\"_channel_bucket\") == bucket).collect()\n               .sort(\"d_channel_key\", \"t\"))\n        if sec.is_empty():\n            continue\n        events.extend(_event_shapes_for_bucket(pl, sec))\n        activity_frames.append(sec.select(\"d_object_key\", pl.col(\"t\").dt.date().alias(\"day\")).unique())\n        last = sec[\"t\"].max()\n        first = sec[\"t\"].min()\n        archive_end = max(last, archive_end) if archive_end is not None else last\n        archive_start = min(first, archive_start) if archive_start is not None else first\n        log(f\"channel bucket {bucket + 1}/{cfg['channel_buckets']}: {sec.height:,} gas seconds\")\n        del sec\n        gc.collect()\n    activity = (pl.concat(activity_frames, how=\"vertical_relaxed\").unique() if activity_frames else\n                pl.DataFrame(schema={\"d_object_key\": pl.String, \"day\": pl.Date}))\n    if archive_end is None:\n        raise ValueError(\"no gas seconds in staged journals\")\n    assert archive_start is not None\n    return events, activity, archive_start, archive_end\n"
}


In [ ]:
TAXONOMY_JSON = '{\n  "version": "3.1",\n  "description": "Классы значений значение_датчика для событийной панели v3. Не метки физических отказов. Источник: справочник_состояний.csv (организаторы) + словарь значений журнала; расхождения со справочником отмечены в notes.",\n  "text_classes": {\n    "link_fault": [\n      "Неисправен"\n    ],\n    "power_off": [\n      "Обесточен"\n    ],\n    "power_on": [\n      "Есть питание",\n      "Питание от сети"\n    ],\n    "power_degraded": [\n      "Питание от батарей",\n      "Батарея разряжена",\n      "Батарея неисправна"\n    ],\n    "device_off": [\n      "Отключено устройство"\n    ],\n    "service": [\n      "Неопределен"\n    ],\n    "undefined_measurement": [\n      "Не определено"\n    ],\n    "object_summary": [\n      "Много неисправных устройств",\n      "Устройства на объекте исправны",\n      "Работают все насосы АНС",\n      "Работают все насосы в АНС",\n      "Включены все насосы АНС"\n    ],\n    "security_mode": [\n      "На охране",\n      "Снято с охраны"\n    ],\n    "flood": [\n      "Затоплен"\n    ],\n    "event_alarm": [\n      "Обнаружено движение",\n      "Движение вверх",\n      "Движение вниз",\n      "Движение влево",\n      "Движение вправо",\n      "Обнаружен дым",\n      "Обнаружен газ",\n      "Не замкнут",\n      "Замкнут",\n      "Рычаг сдернут",\n      "Рычаг сдернут влево",\n      "Рычаг сдернут вправо",\n      "Оба рычага сдернуты",\n      "Вызов",\n      "Разговор",\n      "Температура выше 35ºC1",\n      "Температура выше 40ºC",\n      "Температура выше 27ºC",\n      "Температура ниже 3ºC",\n      "Температура ниже 3ºC1"\n    ],\n    "normal": [\n      "Норма",\n      "Движения нет",\n      "Дыма нет",\n      "Рычаг норма",\n      "Включен",\n      "Выключен",\n      "В норме от +3 до +40",\n      "В норме от +3 до +351",\n      "В норме от +3 до +27ºC"\n    ]\n  },\n  "numeric": {\n    "technical_value_candidates": [\n      -127.0,\n      -100.0,\n      255.0,\n      -3276.0,\n      -3276.8,\n      327.68,\n      3276.7\n    ],\n    "gas_sensor_type": "Газовый датчик",\n    "gas_out_of_range": {\n      "below": 0.0,\n      "above": 100.0\n    },\n    "gas_threshold_percent": 1.0,\n    "gas_warning_percent": 5.0,\n    "gas_band_low": 0.1\n  },\n  "date_like": {\n    "regex": "^\\\\d{2}\\\\.\\\\d{2}\\\\.\\\\d{4}",\n    "clock_1970_prefix": "01.01.1970",\n    "duplicate_timestamp_tolerance_seconds": 5\n  },\n  "notes": [\n    "Неисправен = «только факт потери связи с устройством» (FAQ организаторов); в справочнике есть только у набора 1 (КД Дверь), в журнале — у насосов, вентиляторов, дыма, газа, фаз.",\n    "Обесточен у «Состояния фазы» в справочнике отсутствует; у набора 10 (Датчик дыма) помечен тревожное=false.",\n    "−100, 255 — примеры технических кодов производителей из FAQ, не универсальные пороги; кандидаты проверяются агрегатами на train.",\n    "01.01.1970 — ошибка часов/формата (FAQ: «скорее всего … сбилась дата»), не метка отказа измерителя; даты, совпадающие со временем события, — дубль timestamp.",\n    "Связь: O — только сообщения функции самого канала (норма/событие/число); питание, охрана, затопление, служебные — не O связи.",\n    "Питание: O — только явные «Есть питание»/«Питание от сети»; «Питание от батарей» нейтрально; во время «Неисправен» питание неизвестно (U).",\n    "«Неопределен» в основной версии — U для связи (не O и не восстановление); sensitivity-версия считает его нейтральным (GPT 16/Q1).",\n    "Коды T1a (−127, −100, 255, −3276, −3276.8, 327.68, 3276.7) — фиксированное правило из FAQ и данных 2019 до любой валидации; не означают физический отказ датчика."\n  ],\n  "stream_mapping": {\n    "version": "1.0",\n    "note": "stream_id = (канал, класс сигнала). F — отказ потока, O — однозначное рабочее состояние того же потока. F+O в одной секунде → A (conflict). U-значение: при unknown_overrides=true делает секунду A всегда, иначе — только если в секунде нет F/O этого потока. Значения других классов поток не меняют (GPT 16: O другого класса не закрывает F связи; «Обесточен» не O связи).",\n    "streams": {\n      "link": {\n        "fault": [\n          "link_fault"\n        ],\n        "ok": [\n          "normal",\n          "event_alarm",\n          "numeric"\n        ],\n        "unknown": [\n          "service"\n        ],\n        "unknown_overrides": true\n      },\n      "power": {\n        "fault": [\n          "power_off"\n        ],\n        "ok": [\n          "power_on"\n        ],\n        "unknown": [\n          "link_fault"\n        ],\n        "unknown_overrides": false\n      }\n    },\n    "sensitivity": {\n      "service_neutral": "link.unknown без service: «Неопределен» не меняет состояние связи"\n    }\n  }\n}\n'


In [ ]:
CODE_DIR = Path(tempfile.mkdtemp(prefix="ldt21_code_"))
for name, source in SOURCES.items():
    (CODE_DIR / name).write_text(source, encoding="utf-8")
(CODE_DIR / "state_taxonomy_v3.json").write_text(TAXONOMY_JSON, encoding="utf-8")
sys.path.insert(0, str(CODE_DIR))
import gas_triage_v1 as gt
print("Встроенный код записан:", ", ".join(sorted(SOURCES)))


In [ ]:
# Короткая проверка причинного cutoff на синтетике перед чтением архива
from datetime import datetime, timedelta
t0 = datetime(2024, 12, 31, 23, 55)
def sec(ch, minute, value):
    return {"d_channel_key": ch, "d_object_key": "synthetic",
            "t": t0 + timedelta(minutes=minute), "mn": value, "mx": value}
synthetic = pl.DataFrame([sec("a", -1, 0.2), sec("a", 0, 1.4), sec("a", 4, 0.4),
                          sec("b", -1, 0.2), sec("b", 0, 1.4), sec("b", 4, 0.4)])
shapes = gt._event_shapes_for_bucket(pl, synthetic.sort("d_channel_key", "t"))
activity = synthetic.select("d_object_key", pl.col("t").dt.date().alias("day")).unique()
runs = gt.gf.bounded_silence_runs(pl, synthetic)
labeled = gt.label_events(shapes, activity, runs, t0 + timedelta(hours=1))
assert len(labeled) == 2 and all(e["other_ch_by_live_cutoff"] == 1 for e in labeled)
assert all(e["status_at_day_end"] == "pending_at_day_end" for e in labeled)
assert all(e["triage_retro"] == "censored_by_day" for e in labeled)
print("Синтетический cutoff: OK")


## Preflight


In [ ]:
CONFIG = gt.make_config_21(MODE, {"taxonomy_path": str(CODE_DIR / "state_taxonomy_v3.json")})
PREFLIGHT = gt.preflight(CONFIG)
print("Журналы:", PREFLIGHT["journal_years"])
print("Panel:", PREFLIGHT["panel_manifest_sha256"], "| строк:", PREFLIGHT["panel_rows"],
      "| частей:", PREFLIGHT["panel_parts"])
print("RAM GiB:", PREFLIGHT["ram_gib"], "| свободный диск GiB:", PREFLIGHT["free_stage_gib"],
      "| бакетов:", PREFLIGHT["channel_buckets"])
print(PREFLIGHT["staging_note"])


## Триаж


In [ ]:
RESULT = gt.run_stage21(CONFIG)
print("Статус:", RESULT["status"], "| пересечений:", RESULT["crossings_total"],
      "| пауз:", RESULT["silence_runs"], "| время:", RESULT["runtime_s"], "с")
print("V1:", RESULT["v1_calendar"]["passed"], "| N1:", RESULT["n1_synchrony_control"]["passed"],
      "| backend gate:", RESULT["backend_gate"])


## Агрегатная сводка


In [ ]:
from IPython.display import Markdown, display
OUT = Path(CONFIG["output_dir"])
display(Markdown((OUT / "summary_21_gas_triage_ru.md").read_text(encoding="utf-8")))
for name in ("results_21_gas_triage.json", "summary_21_gas_triage_ru.md"):
    p = OUT / name
    print("-", name, f"({p.stat().st_size / 1024:.1f} КБ)")
